In [ ]:
import pandas as pd

file_path = "../data/raw/bc_card_202601_202606.csv"

bc = pd.read_csv(
    file_path,
    encoding="utf-8-sig"
)

bc.head()

In [ ]:
print("행, 열 개수:", bc.shape)

print("\n컬럼:")
print(bc.columns.tolist())

In [ ]:
bc.info()

In [ ]:
print("기준년월:")
print(sorted(bc["STRD_YYMM"].unique()))

print("\n성별 코드:")
print(bc["GENDER_CD"].value_counts().sort_index())

print("\n연령 코드:")
print(bc["AGE_CD"].value_counts().sort_index())

print("\n시도 개수:", bc["SIDO_NM"].nunique())
print("업종 개수:", bc["TP_BUZ_NM"].nunique())

In [ ]:
pd.crosstab(
    bc["GENDER_CD"],
    bc["AGE_CD"]
)

In [ ]:
bc_person = bc[
    (bc["GENDER_CD"].isin(["1", "2"])) &
    (bc["AGE_CD"].isin(["1", "2", "3", "4", "5", "6"]))
].copy()

print("전체 행:", len(bc))
print("분석용 개인 행:", len(bc_person))

In [ ]:
bc_clean = bc_person.copy()

age_map = {
    "1": "20대이하",
    "2": "20대",
    "3": "30대",
    "4": "40대",
    "5": "50대",
    "6": "60대이상"
}

gender_map = {
    "1": "남성",
    "2": "여성"
}

bc_clean["연령대"] = bc_clean["AGE_CD"].map(age_map)
bc_clean["성별"] = bc_clean["GENDER_CD"].map(gender_map)

bc_clean.head()

In [ ]:
bc_clean = bc_clean.rename(columns={
    "STRD_YYMM": "기준년월",
    "SIDO_NM": "시도",
    "CCG_NM": "시군구",
    "TP_BUZ_NO": "업종코드",
    "TP_BUZ_NM": "업종명",
    "amt": "이용금액",
    "cnt": "이용건수"
})

In [ ]:
bc_clean = bc_clean[
    [
        "기준년월",
        "시도",
        "시군구",
        "성별",
        "연령대",
        "업종코드",
        "업종명",
        "이용금액",
        "이용건수"
    ]
]

bc_clean.head()

In [ ]:
print("중복 행 수:", bc_clean.duplicated().sum())

print("\n업종 목록:")
print(sorted(bc_clean["업종명"].unique()))

print("\n시도 목록:")
print(sorted(bc_clean["시도"].unique()))

In [ ]:
# 지역명의 앞뒤 불필요한 공백 제거
bc_clean["시도"] = bc_clean["시도"].str.strip()
bc_clean["시군구"] = bc_clean["시군구"].str.strip()

# 업종명에 들어간 공백 제거
bc_clean["업종명"] = (
    bc_clean["업종명"]
    .str.replace(r"\s+", "", regex=True)
)

print("업종 개수:", bc_clean["업종명"].nunique())
print(sorted(bc_clean["업종명"].unique()))

In [ ]:
industry_check = (
    bc_clean[["업종코드", "업종명"]]
    .drop_duplicates()
    .sort_values("업종코드")
)

industry_check

In [ ]:
industry_check = (
    bc_clean[["업종코드", "업종명"]]
    .drop_duplicates()
    .sort_values("업종코드")
    .reset_index(drop=True)
)

industry_check

In [ ]:
bc_clean.to_csv(
    "../data/processed/bc_card_clean_202601_202606.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

In [ ]:
bc_age = (
    bc_clean
    .groupby(
        ["시도", "시군구", "업종코드", "업종명", "연령대"],
        as_index=False
    )[["이용금액", "이용건수"]]
    .sum()
)

bc_age.head()

In [ ]:
bc_age["업종총이용금액"] = (
    bc_age
    .groupby(["시도", "시군구", "업종코드"])["이용금액"]
    .transform("sum")
)

bc_age["이용금액비중"] = (
    bc_age["이용금액"]
    / bc_age["업종총이용금액"]
    * 100
).round(1)

bc_age.head(12)

In [ ]:
dominant_age = bc_age.loc[
    bc_age.groupby(
        ["시도", "시군구", "업종코드"]
    )["이용금액비중"].idxmax(),
    [
        "시도",
        "시군구",
        "업종코드",
        "업종명",
        "연령대",
        "이용금액비중",
        "업종총이용금액"
    ]
].copy()

dominant_age = dominant_age.sort_values(
    "이용금액비중",
    ascending=False
).reset_index(drop=True)

dominant_age.head(20)

In [ ]:
dominant_age["업종총이용금액"].describe()

In [ ]:
dominant_age["업종총이용금액"].quantile(
    [0.25, 0.5, 0.75, 0.9]
)

In [ ]:
q75 = dominant_age["업종총이용금액"].quantile(0.75)

candidate = dominant_age[
    dominant_age["업종총이용금액"] >= q75
].copy()

candidate = candidate.sort_values(
    ["이용금액비중", "업종총이용금액"],
    ascending=[False, False]
).reset_index(drop=True)

candidate.head(20)

In [ ]:
for threshold in [40, 50, 60]:
    count = (candidate["이용금액비중"] >= threshold).sum()
    print(f"{threshold}% 이상:", count)

In [ ]:
candidate["연령대"].value_counts()

In [ ]:
candidate["이용금액비중"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95]
)

In [ ]:
monthly_sales = (
    bc_clean
    .groupby(
        ["기준년월", "시도", "시군구", "업종코드", "업종명"],
        as_index=False
    )["이용금액"]
    .sum()
)

monthly_sales.head()

In [ ]:
jan_jun = monthly_sales[
    monthly_sales["기준년월"].isin([202601, 202606])
].pivot_table(
    index=["시도", "시군구", "업종코드", "업종명"],
    columns="기준년월",
    values="이용금액"
).reset_index()

jan_jun["1월대비6월증감률"] = (
    (jan_jun[202606] - jan_jun[202601])
    / jan_jun[202601]
    * 100
).round(1)

jan_jun.head()

In [ ]:
candidate_growth = candidate.merge(
    jan_jun[
        ["시도", "시군구", "업종코드", "업종명", "1월대비6월증감률"]
    ],
    on=["시도", "시군구", "업종코드", "업종명"],
    how="left"
)

candidate_growth.head()

In [ ]:
growth_candidates = candidate_growth[
    candidate_growth["1월대비6월증감률"] > 0
].copy()

growth_candidates = growth_candidates.sort_values(
    ["이용금액비중", "1월대비6월증감률"],
    ascending=[False, False]
).reset_index(drop=True)

growth_candidates.head(20)

In [ ]:
monthly_pivot = monthly_sales.pivot_table(
    index=["시도", "시군구", "업종코드", "업종명"],
    columns="기준년월",
    values="이용금액"
).reset_index()

months = [202601, 202602, 202603, 202604, 202605, 202606]

# 월별 증감률 계산
monthly_pct = monthly_pivot[months].pct_change(axis=1) * 100

# 5번의 월간 변화 평균
monthly_pivot["월평균증감률"] = (
    monthly_pct.iloc[:, 1:]
    .mean(axis=1)
    .round(1)
)

# 전달보다 이용금액이 증가한 달의 수
monthly_pivot["상승개월수"] = (
    monthly_pct.iloc[:, 1:] > 0
).sum(axis=1)

monthly_pivot.head()

In [ ]:
candidate_trend = candidate_growth.merge(
    monthly_pivot[
        [
            "시도",
            "시군구",
            "업종코드",
            "업종명",
            "월평균증감률",
            "상승개월수"
        ]
    ],
    on=["시도", "시군구", "업종코드", "업종명"],
    how="left"
)

candidate_trend.head()

In [ ]:
# 연령 편중도가 상위 10%인 경우를 '고편중'으로 정의
concentration_cut = candidate_trend["이용금액비중"].quantile(0.90)

candidate_trend["고편중여부"] = (
    candidate_trend["이용금액비중"] >= concentration_cut
)

# 6개월 동안 비교적 일관되게 성장했는지 확인
candidate_trend["성장여부"] = (
    (candidate_trend["1월대비6월증감률"] > 0) &
    (candidate_trend["월평균증감률"] > 0) &
    (candidate_trend["상승개월수"] >= 3)
)

In [ ]:
def classify_type(row):
    if row["고편중여부"] and row["성장여부"]:
        return "성장·편중 후보"
    elif row["고편중여부"]:
        return "연령편중 후보"
    elif row["성장여부"]:
        return "현재성장형"
    else:
        return "기타"

candidate_trend["현재유형"] = candidate_trend.apply(
    classify_type,
    axis=1
)

candidate_trend["현재유형"].value_counts()

In [ ]:
growth_concentration = candidate_trend[
    candidate_trend["현재유형"] == "성장·편중 후보"
].copy()

growth_concentration[
    [
        "시도",
        "시군구",
        "업종명",
        "연령대",
        "이용금액비중",
        "1월대비6월증감률",
        "월평균증감률",
        "상승개월수"
    ]
]

In [ ]:
bc_age["비중제곱"] = (
    bc_age["이용금액비중"] / 100
) ** 2

age_concentration = (
    bc_age
    .groupby(
        ["시도", "시군구", "업종코드", "업종명"],
        as_index=False
    )["비중제곱"]
    .sum()
    .rename(columns={
        "비중제곱": "연령집중도"
    })
)

age_concentration.head()

In [ ]:
candidate_trend = candidate_trend.merge(
    age_concentration,
    on=["시도", "시군구", "업종코드", "업종명"],
    how="left"
)

candidate_trend.head()

In [ ]:
bc_profile = dominant_age.merge(
    jan_jun[
        ["시도", "시군구", "업종코드", "업종명", "1월대비6월증감률"]
    ],
    on=["시도", "시군구", "업종코드", "업종명"],
    how="left"
)

bc_profile = bc_profile.merge(
    monthly_pivot[
        [
            "시도",
            "시군구",
            "업종코드",
            "업종명",
            "월평균증감률",
            "상승개월수"
        ]
    ],
    on=["시도", "시군구", "업종코드", "업종명"],
    how="left"
)

bc_profile = bc_profile.merge(
    age_concentration,
    on=["시도", "시군구", "업종코드", "업종명"],
    how="left"
)

bc_profile.head()

In [ ]:
months = [202601, 202602, 202603, 202604, 202605, 202606]

monthly_pivot["데이터보유개월수"] = (
    monthly_pivot[months]
    .notna()
    .sum(axis=1)
)

monthly_pivot["6개월완전"] = (
    monthly_pivot["데이터보유개월수"] == 6
)

monthly_pivot[
    ["시도", "시군구", "업종명", "데이터보유개월수", "6개월완전"]
].head()

In [ ]:
monthly_complete = monthly_pivot[
    monthly_pivot["6개월완전"]
].copy()

In [ ]:
monthly_pivot["6개월완전"].value_counts()

In [ ]:
months = [202601, 202602, 202603, 202604, 202605, 202606]

monthly_pct = (
    monthly_pivot[months]
    .pct_change(axis=1, fill_method=None)
    * 100
)

monthly_pivot["월평균증감률"] = (
    monthly_pct.iloc[:, 1:]
    .mean(axis=1)
    .round(1)
)

monthly_pivot["상승개월수"] = (
    monthly_pct.iloc[:, 1:] > 0
).sum(axis=1)

# 6개월이 완전하지 않은 경우에는
# 성장 관련 값을 사용하지 않도록 NaN 처리
monthly_pivot.loc[
    ~monthly_pivot["6개월완전"],
    ["월평균증감률", "상승개월수"]
] = pd.NA

In [ ]:
bc_profile = bc_profile.merge(
    monthly_pivot[
        [
            "시도",
            "시군구",
            "업종코드",
            "업종명",
            "데이터보유개월수",
            "6개월완전"
        ]
    ],
    on=["시도", "시군구", "업종코드", "업종명"],
    how="left"
)

In [ ]:
# 1월 → 6월 증감률도 monthly_pivot에서 다시 계산
monthly_pivot["1월대비6월증감률"] = (
    (monthly_pivot[202606] - monthly_pivot[202601])
    / monthly_pivot[202601]
    * 100
).round(1)

# 6개월이 완전하지 않으면 성장 관련 값은 사용하지 않음
monthly_pivot.loc[
    ~monthly_pivot["6개월완전"],
    ["1월대비6월증감률", "월평균증감률", "상승개월수"]
] = pd.NA

In [ ]:
bc_profile = dominant_age.merge(
    age_concentration,
    on=["시도", "시군구", "업종코드", "업종명"],
    how="left"
)

bc_profile = bc_profile.merge(
    monthly_pivot[
        [
            "시도",
            "시군구",
            "업종코드",
            "업종명",
            "1월대비6월증감률",
            "월평균증감률",
            "상승개월수",
            "데이터보유개월수",
            "6개월완전"
        ]
    ],
    on=["시도", "시군구", "업종코드", "업종명"],
    how="left"
)

bc_profile.head()

In [ ]:
# 충분한 소비 규모 기준: 전체 지역×업종 중 상위 25%
q75 = bc_profile["업종총이용금액"].quantile(0.75)

analysis_base = bc_profile[
    (bc_profile["6개월완전"] == True) &
    (bc_profile["업종총이용금액"] >= q75)
].copy()

analysis_base = analysis_base.reset_index(drop=True)

analysis_base.head()

In [ ]:
# 분석 대상 중 연령집중도가 상위 10%인 기준
concentration_cut = analysis_base["연령집중도"].quantile(0.90)

analysis_base["고편중여부"] = (
    analysis_base["연령집중도"] >= concentration_cut
)

# 현재 성장 여부
analysis_base["성장여부"] = (
    (analysis_base["1월대비6월증감률"] > 0) &
    (analysis_base["월평균증감률"] > 0) &
    (analysis_base["상승개월수"] >= 3)
)

# 현재 감소 여부
analysis_base["감소여부"] = (
    (analysis_base["1월대비6월증감률"] < 0) &
    (analysis_base["월평균증감률"] < 0) &
    (analysis_base["상승개월수"] <= 2)
)

In [ ]:
def classify_current(row):
    if row["고편중여부"] and row["성장여부"]:
        return "성장·편중형"
    elif row["고편중여부"] and row["감소여부"]:
        return "감소·편중형"
    elif row["고편중여부"]:
        return "연령편중형"
    elif row["성장여부"]:
        return "현재성장형"
    elif row["감소여부"]:
        return "현재감소형"
    else:
        return "안정형"

analysis_base["현재유형"] = analysis_base.apply(
    classify_current,
    axis=1
)

analysis_base["현재유형"].value_counts()

In [ ]:
high_concentration = analysis_base[
    analysis_base["고편중여부"] == True
].copy()

high_concentration.groupby("시도").size().sort_values(ascending=False)

In [ ]:
high_concentration[
    ["시도", "시군구"]
].drop_duplicates().sort_values(
    ["시도", "시군구"]
)

In [ ]:
seoul_candidate = high_concentration[
    high_concentration["시도"] == "서울특별시"
].copy()

seoul_candidate.head()

In [ ]:
future_seoul = future_change.copy()

future_seoul["연령대"] = future_seoul["연령대"].replace({
    "20대미만": "20대이하"
})

future_seoul = future_seoul.rename(columns={
    "자치구별": "시군구"
})

In [ ]:
future_change = pd.read_csv(
    "../data/processed/seoul_future_population_change.csv",
    encoding="utf-8-sig"
)

future_change.head()

In [ ]:
future_seoul = future_change.copy()

future_seoul["연령대"] = future_seoul["연령대"].replace({
    "20대미만": "20대이하"
})

future_seoul = future_seoul.rename(columns={
    "자치구별": "시군구"
})

future_seoul.head()

In [ ]:
# 1. 지역 × 업종 × 연령대별 전체 소비구조
bc_age.to_csv(
    "../data/processed/bc_age_consumption_structure.csv",
    index=False,
    encoding="utf-8-sig"
)

# 2. 전국 지역 × 업종 현재 소비 프로필
bc_profile.to_csv(
    "../data/processed/bc_region_industry_profile.csv",
    index=False,
    encoding="utf-8-sig"
)

# 3. 6개월 완전 + 소비규모 충분한 실제 분석 대상
analysis_base.to_csv(
    "../data/processed/bc_analysis_base.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

In [ ]:
seoul_stress = seoul_candidate.merge(
    future_seoul,
    on=["시군구", "연령대"],
    how="left"
)

seoul_stress.head()

In [ ]:
seoul_stress[
    ["2030 증감률", "2035 증감률", "2040 증감률"]
].isna().sum()

In [ ]:
for year in [2030, 2035, 2040]:
    
    # 인구가 감소하는 경우만 양수로 변환
    seoul_stress[f"{year} 인구감소율"] = (
        -seoul_stress[f"{year} 증감률"]
    ).clip(lower=0)

    # 현재 핵심 고객 의존도 × 미래 인구 감소율
    seoul_stress[f"{year} 소비기반노출도"] = (
        seoul_stress["이용금액비중"]
        * seoul_stress[f"{year} 인구감소율"]
        / 100
    ).round(2)

In [ ]:
seoul_stress[
    [
        "시군구",
        "업종명",
        "연령대",
        "이용금액비중",
        "2035 증감률",
        "2035 소비기반노출도"
    ]
].sort_values(
    "2035 소비기반노출도",
    ascending=False
).head(10)

In [ ]:
(seoul_stress["2035 소비기반노출도"] > 0).value_counts()

In [ ]:
seoul_stress[
    seoul_stress["2035 증감률"] < 0
][
    [
        "시군구",
        "업종명",
        "연령대",
        "이용금액비중",
        "2035 증감률",
        "2035 소비기반노출도"
    ]
].sort_values(
    "2035 소비기반노출도",
    ascending=False
)

In [ ]:
seoul_valid = analysis_base[
    analysis_base["시도"] == "서울특별시"
][
    ["시도", "시군구", "업종코드", "업종명"]
].copy()

In [ ]:
seoul_age_stress = bc_age.merge(
    seoul_valid,
    on=["시도", "시군구", "업종코드", "업종명"],
    how="inner"
)

In [ ]:
seoul_age_stress = seoul_age_stress.merge(
    future_seoul[
        [
            "시군구",
            "연령대",
            "2030 증감률",
            "2035 증감률",
            "2040 증감률"
        ]
    ],
    on=["시군구", "연령대"],
    how="left"
)

In [ ]:
seoul_age_stress.head(12)

In [ ]:
for year in [2030, 2035, 2040]:

    # 각 연령대 소비비중
    share = seoul_age_stress["이용금액비중"] / 100

    # 인구 감소만 위험으로 계산
    population_decline = (
        -seoul_age_stress[f"{year} 증감률"]
    ).clip(lower=0)

    seoul_age_stress[f"{year} 감소노출도"] = (
        share * population_decline
    )

    # 인구 증가/감소를 모두 포함한 순효과
    seoul_age_stress[f"{year} 순인구효과"] = (
        share * seoul_age_stress[f"{year} 증감률"]
    )

In [ ]:
seoul_stress_summary = (
    seoul_age_stress
    .groupby(
        ["시도", "시군구", "업종코드", "업종명"],
        as_index=False
    )[
        [
            "2030 감소노출도",
            "2035 감소노출도",
            "2040 감소노출도",
            "2030 순인구효과",
            "2035 순인구효과",
            "2040 순인구효과"
        ]
    ]
    .sum()
)

seoul_stress_summary.head()

In [ ]:
seoul_result = analysis_base[
    analysis_base["시도"] == "서울특별시"
].merge(
    seoul_stress_summary,
    on=["시도", "시군구", "업종코드", "업종명"],
    how="left"
)

seoul_result.head()

In [ ]:
illusion_growth = seoul_result[
    (seoul_result["성장여부"] == True) &
    (seoul_result["2040 순인구효과"] < 0)
].copy()

illusion_growth = illusion_growth.sort_values(
    "2040 순인구효과",
    ascending=True
).reset_index(drop=True)

illusion_growth[
    [
        "시군구",
        "업종명",
        "연령대",
        "이용금액비중",
        "연령집중도",
        "1월대비6월증감률",
        "월평균증감률",
        "2035 순인구효과",
        "2040 순인구효과",
        "2040 감소노출도"
    ]
].head(20)

In [ ]:
print("착시성장 후보 수:", len(illusion_growth))

In [ ]:
def classify_risk_timing(row):
    if row["2035 순인구효과"] < 0:
        return "2035부터 취약"
    elif row["2040 순인구효과"] < 0:
        return "2040부터 취약"
    else:
        return "비취약"

illusion_growth["위험시점"] = illusion_growth.apply(
    classify_risk_timing,
    axis=1
)

illusion_growth["위험시점"].value_counts()

In [ ]:
illusion_growth[
    [
        "시군구",
        "업종명",
        "연령대",
        "이용금액비중",
        "1월대비6월증감률",
        "2035 순인구효과",
        "2040 순인구효과",
        "2040 감소노출도",
        "위험시점"
    ]
].sort_values(
    ["위험시점", "2040 순인구효과"]
)

In [ ]:
risk_2035 = illusion_growth[
    illusion_growth["2035 순인구효과"] < 0
].copy()

risk_2035 = risk_2035.sort_values(
    "2035 순인구효과",
    ascending=True
).reset_index(drop=True)

risk_2035[
    [
        "시군구",
        "업종명",
        "연령대",
        "이용금액비중",
        "1월대비6월증감률",
        "월평균증감률",
        "2035 순인구효과",
        "2040 순인구효과"
    ]
].head(10)

In [ ]:
target_gu = risk_2035.loc[0, "시군구"]
target_industry = risk_2035.loc[0, "업종명"]

target_age = seoul_age_stress[
    (seoul_age_stress["시군구"] == target_gu) &
    (seoul_age_stress["업종명"] == target_industry)
][
    [
        "연령대",
        "이용금액비중",
        "2035 증감률",
        "2035 순인구효과",
        "2040 증감률",
        "2040 순인구효과"
    ]
].sort_values("2035 순인구효과")

target_age

In [ ]:
# 서울의 신뢰 가능한 지역×업종만 전체 연령대 소비구조와 결합
seoul_age_valid = bc_age.merge(
    seoul_valid,
    on=["시도", "시군구", "업종코드", "업종명"],
    how="inner"
)

# 현재 선택한 업종만 추출
industry_age = seoul_age_valid[
    seoul_age_valid["업종명"] == target_industry
].copy()

# 같은 업종에서 연령대별 소비비중 상위 25% 수준
age_benchmark = (
    industry_age
    .groupby("연령대")["이용금액비중"]
    .quantile(0.75)
    .reset_index()
    .rename(columns={
        "이용금액비중": "동일업종_상위25%비중"
    })
)

age_benchmark

In [ ]:
next_customer = target_age.merge(
    age_benchmark,
    on="연령대",
    how="left"
)

next_customer["확장여력"] = (
    next_customer["동일업종_상위25%비중"]
    - next_customer["이용금액비중"]
).round(1)

next_customer[
    [
        "연령대",
        "이용금액비중",
        "동일업종_상위25%비중",
        "확장여력",
        "2035 증감률"
    ]
]

In [ ]:
# 서울 전체 업종별·연령대별 상위 25% 소비비중 기준
industry_age_benchmark = (
    seoul_age_valid
    .groupby(
        ["업종코드", "업종명", "연령대"]
    )["이용금액비중"]
    .quantile(0.75)
    .reset_index()
    .rename(columns={
        "이용금액비중": "동일업종_상위25%비중"
    })
)

In [ ]:
risk_keys = risk_2035[
    ["시군구", "업종코드", "업종명", "연령대"]
].copy()

risk_keys = risk_keys.rename(
    columns={"연령대": "현재핵심연령대"}
)

next_pool = seoul_age_valid.merge(
    risk_keys,
    on=["시군구", "업종코드", "업종명"],
    how="inner"
)

next_pool = next_pool.merge(
    industry_age_benchmark,
    on=["업종코드", "업종명", "연령대"],
    how="left"
)

next_pool = next_pool.merge(
    future_seoul[
        ["시군구", "연령대", "2035 증감률"]
    ],
    on=["시군구", "연령대"],
    how="left"
)

next_pool["확장여력"] = (
    next_pool["동일업종_상위25%비중"]
    - next_pool["이용금액비중"]
).round(1)

In [ ]:
next_customer_candidates = next_pool[
    (next_pool["연령대"] != next_pool["현재핵심연령대"]) &
    (next_pool["확장여력"] > 0) &
    (next_pool["2035 증감률"] > 0)
].copy()

next_customer_candidates[
    [
        "시군구",
        "업종명",
        "현재핵심연령대",
        "연령대",
        "이용금액비중",
        "확장여력",
        "2035 증감률"
    ]
].sort_values(
    ["확장여력", "2035 증감률"],
    ascending=False
)

In [ ]:
core_growth = future_seoul[
    ["시군구", "연령대", "2035 증감률"]
].rename(columns={
    "연령대": "현재핵심연령대",
    "2035 증감률": "핵심연령_2035증감률"
})

next_whatif = next_customer_candidates.merge(
    core_growth,
    on=["시군구", "현재핵심연령대"],
    how="left"
)

In [ ]:
current_effect = seoul_result[
    ["시군구", "업종코드", "업종명", "2035 순인구효과"]
]

next_whatif = next_whatif.merge(
    current_effect,
    on=["시군구", "업종코드", "업종명"],
    how="left"
)

In [ ]:
next_whatif["1pp전환효과"] = (
    next_whatif["2035 증감률"]
    - next_whatif["핵심연령_2035증감률"]
) / 100

In [ ]:
next_whatif["중립화필요pp"] = (
    -next_whatif["2035 순인구효과"]
    / next_whatif["1pp전환효과"]
)

next_whatif.loc[
    next_whatif["2035 순인구효과"] >= 0,
    "중립화필요pp"
] = 0

next_whatif["중립화필요pp"] = (
    next_whatif["중립화필요pp"]
    .round(1)
)

In [ ]:
next_whatif["전환가능여부"] = (
    next_whatif["중립화필요pp"]
    <= next_whatif["확장여력"]
)

In [ ]:
next_whatif[
    [
        "시군구",
        "업종명",
        "현재핵심연령대",
        "연령대",
        "확장여력",
        "핵심연령_2035증감률",
        "2035 증감률",
        "2035 순인구효과",
        "중립화필요pp",
        "전환가능여부"
    ]
].sort_values("중립화필요pp")

In [ ]:
# 동일업종 상위 25% 수준까지를 현실적인 전환 목표로 사용
next_whatif["벤치마크전환pp"] = (
    next_whatif["확장여력"]
    .clip(lower=0)
)

# 그만큼 고객구조를 전환했을 때 2035 순인구효과
next_whatif["전환후_2035순인구효과"] = (
    next_whatif["2035 순인구효과"]
    + next_whatif["벤치마크전환pp"]
      * next_whatif["1pp전환효과"]
).round(2)

# 위험이 얼마나 줄어드는지
next_whatif["위험완화율"] = (
    (
        next_whatif["전환후_2035순인구효과"]
        - next_whatif["2035 순인구효과"]
    )
    / abs(next_whatif["2035 순인구효과"])
    * 100
).round(1)

next_whatif[
    [
        "시군구",
        "업종명",
        "현재핵심연령대",
        "연령대",
        "확장여력",
        "2035 순인구효과",
        "전환후_2035순인구효과",
        "위험완화율"
    ]
].sort_values(
    "위험완화율",
    ascending=False
)

In [ ]:
core_future = future_seoul[
    ["시군구", "연령대", "2035 증감률"]
].rename(columns={
    "연령대": "현재핵심연령대",
    "2035 증감률": "핵심연령_2035증감률"
})

next_pool_v2 = next_pool.merge(
    core_future,
    on=["시군구", "현재핵심연령대"],
    how="left"
)

next_pool_v2["인구상대우위"] = (
    next_pool_v2["2035 증감률"]
    - next_pool_v2["핵심연령_2035증감률"]
).round(1)

next_candidates_v2 = next_pool_v2[
    (next_pool_v2["연령대"] != next_pool_v2["현재핵심연령대"]) &
    (next_pool_v2["확장여력"] > 0) &
    (next_pool_v2["인구상대우위"] > 0)
].copy()

In [ ]:
next_candidates_v2[
    [
        "시군구",
        "업종명",
        "현재핵심연령대",
        "연령대",
        "확장여력",
        "핵심연령_2035증감률",
        "2035 증감률",
        "인구상대우위"
    ]
].sort_values(
    ["인구상대우위", "확장여력"],
    ascending=False
)

In [ ]:
# Next Customer로 1%p 전환했을 때
# 2035 인구구조 충격이 얼마나 개선되는지
next_candidates_v2["1pp전환효과"] = (
    next_candidates_v2["인구상대우위"] / 100
)

# 동일업종 상위 25% 수준까지 전환할 경우
# 얻을 수 있는 최대 개선 효과
next_candidates_v2["벤치마크최대개선"] = (
    next_candidates_v2["확장여력"]
    * next_candidates_v2["1pp전환효과"]
).round(2)

In [ ]:
next_candidates_v2 = next_candidates_v2.merge(
    seoul_result[
        ["시군구", "업종코드", "업종명", "2035 순인구효과"]
    ],
    on=["시군구", "업종코드", "업종명"],
    how="left"
)

next_candidates_v2["전환후_2035효과"] = (
    next_candidates_v2["2035 순인구효과"]
    + next_candidates_v2["벤치마크최대개선"]
).round(2)

next_candidates_v2["위험완화율"] = (
    next_candidates_v2["벤치마크최대개선"]
    / abs(next_candidates_v2["2035 순인구효과"])
    * 100
).round(1)

In [ ]:
best_next_customer = next_candidates_v2.loc[
    next_candidates_v2.groupby(
        ["시군구", "업종코드"]
    )["벤치마크최대개선"].idxmax()
].copy()

best_next_customer = best_next_customer.sort_values(
    "위험완화율",
    ascending=False
).reset_index(drop=True)

best_next_customer[
    [
        "시군구",
        "업종명",
        "현재핵심연령대",
        "연령대",
        "확장여력",
        "인구상대우위",
        "2035 순인구효과",
        "전환후_2035효과",
        "위험완화율"
    ]
]

In [ ]:
# 현재 핵심고객과 성장정보
current_info = seoul_result[
    [
        "시군구",
        "업종코드",
        "업종명",
        "연령대",
        "이용금액비중",
        "1월대비6월증감률",
        "월평균증감률",
        "상승개월수"
    ]
].rename(columns={
    "연령대": "현재핵심연령대",
    "이용금액비중": "현재핵심고객비중"
})

solution_base = best_next_customer.merge(
    current_info,
    on=["시군구", "업종코드", "업종명", "현재핵심연령대"],
    how="left"
)

solution_base = solution_base.rename(columns={
    "연령대": "NextCustomer",
    "이용금액비중": "현재NextCustomer비중",
    "확장여력": "권장전환pp"
})

solution_base["NextCustomer목표비중"] = (
    solution_base["현재NextCustomer비중"]
    + solution_base["권장전환pp"]
).round(1)

In [ ]:
solution_base[
    [
        "시군구",
        "업종명",
        "현재핵심연령대",
        "현재핵심고객비중",
        "NextCustomer",
        "현재NextCustomer비중",
        "권장전환pp",
        "NextCustomer목표비중",
        "1월대비6월증감률",
        "2035 순인구효과",
        "전환후_2035효과",
        "위험완화율"
    ]
]

In [ ]:
solution_base.to_csv(
    "../data/processed/seoul_next_customer_solution.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

In [ ]:
target_solution = (
    solution_base
    .sort_values("위험완화율", ascending=False)
    .iloc[0]
)

target_solution[
    [
        "시군구",
        "업종명",
        "현재핵심연령대",
        "NextCustomer",
        "권장전환pp",
        "NextCustomer목표비중",
        "2035 순인구효과",
        "전환후_2035효과",
        "위험완화율"
    ]
]

In [ ]:
expansion_customer = next_candidates_v2.loc[
    next_candidates_v2.groupby(
        ["시군구", "업종코드"]
    )["확장여력"].idxmax()
].copy()

expansion_customer = expansion_customer.sort_values(
    "확장여력",
    ascending=False
).reset_index(drop=True)

expansion_customer[
    [
        "시군구",
        "업종명",
        "현재핵심연령대",
        "연령대",
        "확장여력",
        "인구상대우위"
    ]
]

In [ ]:
defense_customer = best_next_customer[
    [
        "시군구",
        "업종코드",
        "업종명",
        "현재핵심연령대",
        "연령대",
        "위험완화율"
    ]
].copy()

defense_customer = defense_customer.rename(columns={
    "연령대": "방어형고객"
})

expansion_customer_clean = expansion_customer[
    [
        "시군구",
        "업종코드",
        "업종명",
        "연령대",
        "확장여력"
    ]
].copy()

expansion_customer_clean = expansion_customer_clean.rename(columns={
    "연령대": "확장형고객"
})

In [ ]:
customer_strategy = defense_customer.merge(
    expansion_customer_clean,
    on=["시군구", "업종코드", "업종명"],
    how="left"
)

customer_strategy.head()

In [ ]:
customer_strategy["전략일치"] = (
    customer_strategy["방어형고객"]
    == customer_strategy["확장형고객"]
)

customer_strategy["전략일치"].value_counts()

In [ ]:
next_ranked = next_candidates_v2.copy()

next_ranked["후보순위"] = (
    next_ranked
    .groupby(["시군구", "업종코드"])["벤치마크최대개선"]
    .rank(method="first", ascending=False)
    .astype(int)
)

next_top2 = next_ranked[
    next_ranked["후보순위"] <= 2
].copy()

next_top2 = next_top2.sort_values(
    ["시군구", "업종코드", "후보순위"]
)

next_top2[
    [
        "시군구",
        "업종명",
        "현재핵심연령대",
        "후보순위",
        "연령대",
        "확장여력",
        "인구상대우위",
        "벤치마크최대개선"
    ]
]

In [ ]:
next_pool_v2[
    next_pool_v2["연령대"].isin(["20대이하", "20대"])
][
    [
        "시군구",
        "업종명",
        "현재핵심연령대",
        "연령대",
        "확장여력",
        "핵심연령_2035증감률",
        "2035 증감률",
        "인구상대우위"
    ]
]

In [ ]:
(next_pool_v2["확장여력"] > 0) &
(next_pool_v2["인구상대우위"] > 0)

In [ ]:
market_candidates = next_pool_v2[
    (next_pool_v2["연령대"] != next_pool_v2["현재핵심연령대"]) &
    (next_pool_v2["확장여력"] > 0)
].copy()

market_candidates.head()

In [ ]:
def customer_type(row):
    if row["2035 증감률"] > 0:
        return "성장·방어형"
    elif row["인구상대우위"] > 0:
        return "상대안정형"
    else:
        return "시장확장형"

market_candidates["후보유형"] = market_candidates.apply(
    customer_type,
    axis=1
)

market_candidates["후보유형"].value_counts()


In [ ]:
pd.crosstab(
    market_candidates["후보유형"],
    market_candidates["연령대"]
)

In [ ]:
market_candidates["방어개선잠재력"] = (
    market_candidates["확장여력"]
    * market_candidates["인구상대우위"].clip(lower=0)
    / 100
).round(2)

In [ ]:
defense_pool = market_candidates[
    market_candidates["후보유형"].isin(
        ["성장·방어형", "상대안정형"]
    )
].copy()

defense_best = defense_pool.loc[
    defense_pool.groupby(
        ["시군구", "업종코드"]
    )["방어개선잠재력"].idxmax()
].copy()

defense_best = defense_best[
    [
        "시군구",
        "업종코드",
        "업종명",
        "현재핵심연령대",
        "연령대",
        "후보유형",
        "확장여력",
        "인구상대우위",
        "방어개선잠재력"
    ]
].rename(columns={
    "연령대": "방어형고객"
})

In [ ]:
expansion_pool = market_candidates[
    market_candidates["후보유형"] == "시장확장형"
].copy()

expansion_best = expansion_pool.loc[
    expansion_pool.groupby(
        ["시군구", "업종코드"]
    )["확장여력"].idxmax()
].copy()

expansion_best = expansion_best[
    [
        "시군구",
        "업종코드",
        "업종명",
        "연령대",
        "확장여력"
    ]
].rename(columns={
    "연령대": "확장형고객",
    "확장여력": "확장형확장여력"
})

In [ ]:
customer_solution = defense_best.merge(
    expansion_best,
    on=["시군구", "업종코드", "업종명"],
    how="outer"
)

customer_solution.head()

In [ ]:
solution_keys = risk_2035[
    ["시군구", "업종코드", "업종명", "연령대"]
].drop_duplicates().rename(columns={
    "연령대": "현재핵심연령대"
})

In [ ]:
defense_result = defense_best[
    [
        "시군구",
        "업종코드",
        "업종명",
        "방어형고객",
        "후보유형",
        "확장여력",
        "인구상대우위",
        "방어개선잠재력"
    ]
].copy()

In [ ]:
expansion_result = expansion_best[
    [
        "시군구",
        "업종코드",
        "업종명",
        "확장형고객",
        "확장형확장여력"
    ]
].copy()

In [ ]:
customer_solution = solution_keys.merge(
    defense_result,
    on=["시군구", "업종코드", "업종명"],
    how="left"
)

customer_solution = customer_solution.merge(
    expansion_result,
    on=["시군구", "업종코드", "업종명"],
    how="left"
)

customer_solution.head()

In [ ]:
customer_solution = customer_solution.rename(columns={
    "후보유형": "방어형유형",
    "확장여력": "방어형확장여력"
})

In [ ]:
customer_solution.to_csv(
    "../data/processed/seoul_customer_strategy.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

In [ ]:
retail_industries = [
    "대형할인점",
    "편의점",
    "슈퍼마켓"
]

food_industries = [
    "일반한식",
    "갈비전문점",
    "한정식",
    "일식회집",
    "중국음식",
    "서양음식",
    "스낵",
    "제과점"
]

In [ ]:
seoul_result = analysis_base[
    analysis_base["시도"] == "서울특별시"
].merge(
    seoul_stress_summary,
    on=["시도", "시군구", "업종코드", "업종명"],
    how="left"
)

seoul_result.head()

In [ ]:
startup_candidates = seoul_result[
    (seoul_result["6개월완전"] == True) &
    (seoul_result["2035 순인구효과"] >= 0)
].copy()

startup_candidates = startup_candidates.sort_values(
    ["2035 순인구효과", "월평균증감률"],
    ascending=[False, False]
)

startup_candidates[
    [
        "시군구",
        "업종명",
        "연령대",
        "이용금액비중",
        "연령집중도",
        "월평균증감률",
        "2035 순인구효과",
        "2040 순인구효과"
    ]
].head(20)

In [ ]:
startup_candidates["성장성점수"] = (
    startup_candidates["월평균증감률"]
    .rank(pct=True)
    * 100
).round(1)

startup_candidates["미래안정성점수"] = (
    startup_candidates["2035 순인구효과"]
    .rank(pct=True)
    * 100
).round(1)

# 연령집중도는 낮을수록 좋으므로 ascending=False
startup_candidates["고객다양성점수"] = (
    startup_candidates["연령집중도"]
    .rank(pct=True, ascending=False)
    * 100
).round(1)

In [ ]:
startup_candidates["미래창업기회점수"] = (
    (
        startup_candidates["성장성점수"]
        + startup_candidates["미래안정성점수"]
        + startup_candidates["고객다양성점수"]
    ) / 3
).round(1)

startup_ranking = startup_candidates.sort_values(
    "미래창업기회점수",
    ascending=False
).reset_index(drop=True)

In [ ]:
startup_ranking[
    [
        "시군구",
        "업종명",
        "성장성점수",
        "미래안정성점수",
        "고객다양성점수",
        "미래창업기회점수"
    ]
].head(20)

In [ ]:
# 2040년 미래 안정성도 0~100점으로 변환
startup_candidates["2040안정성점수"] = (
    startup_candidates["2040 순인구효과"]
    .rank(pct=True)
    * 100
).round(1)

# 2035 + 2040을 함께 반영
startup_candidates["미래안정성점수"] = (
    (
        startup_candidates["2035 순인구효과"].rank(pct=True) * 100
        + startup_candidates["2040 순인구효과"].rank(pct=True) * 100
    ) / 2
).round(1)

# 최종 점수 다시 계산
startup_candidates["미래창업기회점수"] = (
    (
        startup_candidates["성장성점수"]
        + startup_candidates["미래안정성점수"]
        + startup_candidates["고객다양성점수"]
    ) / 3
).round(1)

In [ ]:
def startup_type(row):
    if row["2040 순인구효과"] >= 0:
        return "미래지속형"
    else:
        return "중기안정형"

startup_candidates["창업유형"] = startup_candidates.apply(
    startup_type,
    axis=1
)

In [ ]:
startup_ranking = startup_candidates.sort_values(
    "미래창업기회점수",
    ascending=False
).reset_index(drop=True)

startup_ranking[
    [
        "시군구",
        "업종명",
        "성장성점수",
        "미래안정성점수",
        "고객다양성점수",
        "미래창업기회점수",
        "창업유형"
    ]
].head(20)

In [ ]:
startup_ranking["지역내순위"] = (
    startup_ranking
    .groupby("시군구")["미래창업기회점수"]
    .rank(method="first", ascending=False)
    .astype(int)
)

startup_top3 = startup_ranking[
    startup_ranking["지역내순위"] <= 3
].copy()

startup_top3 = startup_top3.sort_values(
    ["시군구", "지역내순위"]
)

startup_top3[
    [
        "시군구",
        "지역내순위",
        "업종명",
        "성장성점수",
        "미래안정성점수",
        "고객다양성점수",
        "미래창업기회점수",
        "창업유형"
    ]
]

In [ ]:
startup_top3.to_csv(
    "../data/processed/seoul_startup_top3.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

In [ ]:
owner_strategy_v2 = customer_solution.merge(
    seoul_result[
        [
            "시군구",
            "업종코드",
            "업종명",
            "2035 순인구효과",
            "2040 순인구효과"
        ]
    ],
    on=["시군구", "업종코드", "업종명"],
    how="left"
)

owner_strategy_v2.head()

In [ ]:
def classify_long_term(row):
    if row["2040 순인구효과"] >= 0:
        return "장기회복형"
    elif row["2040 순인구효과"] < row["2035 순인구효과"]:
        return "장기악화형"
    else:
        return "장기개선형"

owner_strategy_v2["장기위험유형"] = owner_strategy_v2.apply(
    classify_long_term,
    axis=1
)

owner_strategy_v2[
    [
        "시군구",
        "업종명",
        "현재핵심연령대",
        "방어형고객",
        "확장형고객",
        "2035 순인구효과",
        "2040 순인구효과",
        "장기위험유형"
    ]
]

In [ ]:
# 현재 핵심고객의 2040 인구변화
core_2040 = future_seoul[
    ["시군구", "연령대", "2040 증감률"]
].rename(columns={
    "연령대": "현재핵심연령대",
    "2040 증감률": "핵심고객_2040증감률"
})

owner_strategy_v2 = owner_strategy_v2.merge(
    core_2040,
    on=["시군구", "현재핵심연령대"],
    how="left"
)

In [ ]:
defense_2040 = future_seoul[
    ["시군구", "연령대", "2040 증감률"]
].rename(columns={
    "연령대": "방어형고객",
    "2040 증감률": "방어형고객_2040증감률"
})

owner_strategy_v2 = owner_strategy_v2.merge(
    defense_2040,
    on=["시군구", "방어형고객"],
    how="left"
)

In [ ]:
owner_strategy_v2["2040방어상대우위"] = (
    owner_strategy_v2["방어형고객_2040증감률"]
    - owner_strategy_v2["핵심고객_2040증감률"]
).round(1)

owner_strategy_v2["2040방어유효"] = (
    owner_strategy_v2["2040방어상대우위"] > 0
)

In [ ]:
owner_strategy_v2[
    [
        "시군구",
        "업종명",
        "현재핵심연령대",
        "방어형고객",
        "핵심고객_2040증감률",
        "방어형고객_2040증감률",
        "2040방어상대우위",
        "2040방어유효"
    ]
]

In [ ]:
def defense_status(row):
    if pd.isna(row["방어형고객"]):
        return "방어형 후보 없음"
    elif row["2040방어유효"]:
        return "장기유효"
    else:
        return "장기재검토"

owner_strategy_v2["방어전략상태"] = owner_strategy_v2.apply(
    defense_status,
    axis=1
)

owner_strategy_v2["방어전략상태"].value_counts()

In [ ]:
owner_strategy_v2["2040_1pp전환효과"] = (
    owner_strategy_v2["2040방어상대우위"] / 100
)

owner_strategy_v2["2040_최대개선효과"] = (
    owner_strategy_v2["방어형확장여력"]
    * owner_strategy_v2["2040_1pp전환효과"]
).round(2)

owner_strategy_v2["전환후_2040순인구효과"] = (
    owner_strategy_v2["2040 순인구효과"]
    + owner_strategy_v2["2040_최대개선효과"]
).round(2)

In [ ]:
owner_strategy_v2[
    [
        "시군구",
        "업종명",
        "현재핵심연령대",
        "방어형고객",
        "방어전략상태",
        "2035 순인구효과",
        "2040 순인구효과",
        "전환후_2040순인구효과"
    ]
]

In [ ]:
owner_strategy_v2["2040위험완화율"] = (
    (
        owner_strategy_v2["전환후_2040순인구효과"]
        - owner_strategy_v2["2040 순인구효과"]
    )
    / abs(owner_strategy_v2["2040 순인구효과"])
    * 100
).round(1)

In [ ]:
owner_strategy_v2[
    [
        "시군구",
        "업종명",
        "현재핵심연령대",
        "방어형고객",
        "방어전략상태",
        "2040 순인구효과",
        "전환후_2040순인구효과",
        "2040위험완화율"
    ]
]

In [ ]:
owner_strategy_v2.to_csv(
    "../data/processed/seoul_owner_strategy_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

In [ ]:
# 1. 고객 전환 효과가 큰 장기유효 사례
valid_cases = (
    owner_strategy_v2[
        owner_strategy_v2["방어전략상태"] == "장기유효"
    ]
    .sort_values("2040위험완화율", ascending=False)
)

# 2. 2035 전략이 2040에는 재검토가 필요한 사례
review_cases = owner_strategy_v2[
    owner_strategy_v2["방어전략상태"] == "장기재검토"
]

# 3. 적절한 방어형 고객을 찾지 못한 사례
no_defense_cases = owner_strategy_v2[
    owner_strategy_v2["방어전략상태"] == "방어형 후보 없음"
]

In [ ]:
valid_cases[
    ["시군구", "업종명", "현재핵심연령대", "방어형고객", "2040위험완화율"]
].head(5)

In [ ]:
review_cases[
    ["시군구", "업종명", "현재핵심연령대", "방어형고객"]
]

In [ ]:
no_defense_cases[
    ["시군구", "업종명", "현재핵심연령대"]
]

In [ ]:
case_success = (
    owner_strategy_v2[
        owner_strategy_v2["방어전략상태"] == "장기유효"
    ]
    .sort_values("2040위험완화율", ascending=False)
    .iloc[0]
)

case_success[
    [
        "시군구",
        "업종명",
        "현재핵심연령대",
        "방어형고객",
        "2035 순인구효과",
        "2040 순인구효과",
        "전환후_2040순인구효과",
        "2040위험완화율"
    ]
]

In [ ]:
case_success_2035_after = (
    case_success["2035 순인구효과"]
    + case_success["방어개선잠재력"]
)

print("2035 전환 전:", case_success["2035 순인구효과"])
print("2035 전환 후:", round(case_success_2035_after, 2))

print("2040 전환 전:", case_success["2040 순인구효과"])
print("2040 전환 후:", case_success["전환후_2040순인구효과"])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

years = ["2035", "2040"]

before = [
    case_success["2035 순인구효과"],
    case_success["2040 순인구효과"]
]

after = [
    case_success_2035_after,
    case_success["전환후_2040순인구효과"]
]

x = np.arange(len(years))
width = 0.35

plt.figure(figsize=(7, 5))

plt.bar(x - width/2, before, width, label="전환 전")
plt.bar(x + width/2, after, width, label="전환 후")

plt.axhline(0, linewidth=1)

plt.xticks(x, years)
plt.ylabel("순인구효과")
plt.title(
    f'{case_success["시군구"]} {case_success["업종명"]}\n'
    f'{case_success["현재핵심연령대"]} → {case_success["방어형고객"]} 전환 효과'
)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 한글 폰트 설정
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

years = ["2035", "2040"]

before = [
    case_success["2035 순인구효과"],
    case_success["2040 순인구효과"]
]

after = [
    case_success_2035_after,
    case_success["전환후_2040순인구효과"]
]

x = np.arange(len(years))
width = 0.35

plt.figure(figsize=(7, 5))

plt.bar(x - width/2, before, width, label="전환 전")
plt.bar(x + width/2, after, width, label="전환 후")

plt.axhline(0, linewidth=1)

plt.xticks(x, years)
plt.ylabel("순인구효과")
plt.title(
    f'{case_success["시군구"]} {case_success["업종명"]}\n'
    f'{case_success["현재핵심연령대"]} → {case_success["방어형고객"]} 전환 효과'
)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
case_age_detail = seoul_age_stress[
    (seoul_age_stress["시군구"] == case_success["시군구"]) &
    (seoul_age_stress["업종명"] == case_success["업종명"])
].copy()

case_age_detail = case_age_detail.sort_values("연령대")

plt.figure(figsize=(8, 5))

plt.bar(
    case_age_detail["연령대"],
    case_age_detail["2040 순인구효과"]
)

plt.axhline(0, linewidth=1)

plt.ylabel("2040 순인구효과")
plt.title(
    f'{case_success["시군구"]} {case_success["업종명"]}\n'
    '연령대별 2040 인구구조 영향'
)

plt.tight_layout()
plt.show()

In [ ]:
age_order = [
    "20대이하",
    "20대",
    "30대",
    "40대",
    "50대",
    "60대이상"
]

case_age_detail["연령대"] = pd.Categorical(
    case_age_detail["연령대"],
    categories=age_order,
    ordered=True
)

case_age_detail = case_age_detail.sort_values("연령대")

plt.figure(figsize=(8, 5))

bars = plt.bar(
    case_age_detail["연령대"],
    case_age_detail["2040 순인구효과"]
)

plt.axhline(0, linewidth=1)

plt.ylabel("2040 순인구효과")
plt.title(
    f'{case_success["시군구"]} {case_success["업종명"]}\n'
    '연령대별 2040 인구구조 영향'
)

plt.tight_layout()
plt.show()

In [ ]:
# 미래창업기회점수가 가장 높은 후보의 지역 선택
target_startup_gu = (
    startup_top3
    .sort_values("미래창업기회점수", ascending=False)
    .iloc[0]["시군구"]
)

startup_case = (
    startup_top3[
        startup_top3["시군구"] == target_startup_gu
    ]
    .sort_values("지역내순위")
    .copy()
)

startup_case[
    [
        "시군구",
        "지역내순위",
        "업종명",
        "성장성점수",
        "미래안정성점수",
        "고객다양성점수",
        "미래창업기회점수",
        "창업유형"
    ]
]

In [ ]:
plt.figure(figsize=(8, 5))

plt.barh(
    startup_case["업종명"],
    startup_case["미래창업기회점수"]
)

plt.xlabel("미래창업기회점수")
plt.title(
    f"{target_startup_gu} 예비창업자 추천 업종 TOP 3"
)

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
x = np.arange(len(startup_case["업종명"]))
width = 0.25

plt.figure(figsize=(9, 5))

plt.bar(
    x - width,
    startup_case["성장성점수"],
    width,
    label="현재 성장성"
)

plt.bar(
    x,
    startup_case["미래안정성점수"],
    width,
    label="미래 안정성"
)

plt.bar(
    x + width,
    startup_case["고객다양성점수"],
    width,
    label="고객 다양성"
)

plt.xticks(x, startup_case["업종명"])
plt.ylabel("상대점수")
plt.ylim(0, 100)

plt.title(
    f"{target_startup_gu} 추천 업종 TOP 3\n평가요소 비교"
)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.savefig(
    "../output/startup_top3_factor_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.legend()
plt.tight_layout()

plt.savefig(
    "../output/owner_transition_effect.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.savefig(
    "../output/owner_age_2040_effect.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

years = ["2035", "2040"]

before = [
    case_success["2035 순인구효과"],
    case_success["2040 순인구효과"]
]

after = [
    case_success_2035_after,
    case_success["전환후_2040순인구효과"]
]

x = np.arange(len(years))
width = 0.35

plt.figure(figsize=(7, 5))

plt.bar(x - width/2, before, width, label="전환 전")
plt.bar(x + width/2, after, width, label="전환 후")

plt.axhline(0, linewidth=1)

plt.xticks(x, years)
plt.ylabel("순인구효과")

plt.title(
    f'{case_success["시군구"]} {case_success["업종명"]}\n'
    f'{case_success["현재핵심연령대"]} → {case_success["방어형고객"]} 전환 효과'
)

plt.legend()
plt.tight_layout()

# 저장은 반드시 show 전에
plt.savefig(
    "../output/owner_transition_effect.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 대표 사례의 연령대별 데이터 가져오기
case_age_detail = seoul_age_stress[
    (seoul_age_stress["시군구"] == case_success["시군구"]) &
    (seoul_age_stress["업종명"] == case_success["업종명"])
].copy()

# 연령대 순서 정리
age_order = [
    "20대이하",
    "20대",
    "30대",
    "40대",
    "50대",
    "60대이상"
]

case_age_detail["연령대"] = pd.Categorical(
    case_age_detail["연령대"],
    categories=age_order,
    ordered=True
)

case_age_detail = case_age_detail.sort_values("연령대")

# 그래프 그리기
plt.figure(figsize=(8, 5))

plt.bar(
    case_age_detail["연령대"],
    case_age_detail["2040 순인구효과"]
)

plt.axhline(0, linewidth=1)

plt.ylabel("2040 순인구효과")

plt.title(
    f'{case_success["시군구"]} {case_success["업종명"]}\n'
    '연령대별 2040 인구구조 영향'
)

plt.tight_layout()

# 이미지 저장
plt.savefig(
    "../output/owner_age_2040_effect.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 미래창업기회점수가 가장 높은 후보의 지역 선택
target_startup_gu = (
    startup_top3
    .sort_values("미래창업기회점수", ascending=False)
    .iloc[0]["시군구"]
)

# 해당 지역 TOP 3
startup_case = (
    startup_top3[
        startup_top3["시군구"] == target_startup_gu
    ]
    .sort_values("지역내순위")
    .copy()
)

# 그래프
plt.figure(figsize=(8, 5))

plt.barh(
    startup_case["업종명"],
    startup_case["미래창업기회점수"]
)

plt.xlabel("미래창업기회점수")
plt.title(
    f"{target_startup_gu} 예비창업자 추천 업종 TOP 3"
)

plt.gca().invert_yaxis()

plt.tight_layout()

# 저장
plt.savefig(
    "../output/startup_top3.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

x = np.arange(len(startup_case["업종명"]))
width = 0.25

plt.figure(figsize=(9, 5))

plt.bar(
    x - width,
    startup_case["성장성점수"],
    width,
    label="현재 성장성"
)

plt.bar(
    x,
    startup_case["미래안정성점수"],
    width,
    label="미래 안정성"
)

plt.bar(
    x + width,
    startup_case["고객다양성점수"],
    width,
    label="고객 다양성"
)

plt.xticks(
    x,
    startup_case["업종명"]
)

plt.ylabel("후보군 내 상대점수")
plt.ylim(0, 100)

plt.title(
    f"{target_startup_gu} 추천 업종 TOP 3\n평가요소 비교"
)

plt.legend()
plt.tight_layout()

# 저장
plt.savefig(
    "../output/startup_top3_factor_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 서울 분석 대상
illusion_plot = seoul_result[
    seoul_result["6개월완전"] == True
].copy()

plt.figure(figsize=(9, 6))

plt.scatter(
    illusion_plot["월평균증감률"],
    illusion_plot["2035 순인구효과"],
    alpha=0.6
)

# 기준선
plt.axvline(0, linewidth=1)
plt.axhline(0, linewidth=1)

plt.xlabel("현재 월평균 소비 증감률(%)")
plt.ylabel("2035 미래 소비기반 스트레스")
plt.title(
    "현재 성장성과 미래 소비기반의 불일치\n"
    "오른쪽 아래 = 현재는 성장하지만 미래 고객기반은 악화"
)

plt.tight_layout()

plt.savefig(
    "../output/illusion_growth_scatter.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
illusion_examples = (
    seoul_result[
        (seoul_result["월평균증감률"] > 0) &
        (seoul_result["2035 순인구효과"] < 0)
    ]
    .sort_values(
        ["2035 순인구효과", "월평균증감률"],
        ascending=[True, False]
    )
    .head(10)
    .copy()
)

illusion_examples[
    [
        "시군구",
        "업종명",
        "연령대",
        "월평균증감률",
        "2035 순인구효과",
        "2040 순인구효과"
    ]
]

In [ ]:
# 현재 성장 중인 서울 지역×업종
current_growth = seoul_result[
    seoul_result["월평균증감률"] > 0
].copy()

# 현재는 성장하지만 2035 미래 소비기반은 악화
illusion_2035 = current_growth[
    current_growth["2035 순인구효과"] < 0
].copy()

print("현재 성장 사례 수 :", len(current_growth))
print("2035 착시성장 사례 수 :", len(illusion_2035))

print(
    "착시성장 비율 :",
    round(len(illusion_2035) / len(current_growth) * 100, 1),
    "%"
)

In [ ]:
illusion_industry = (
    illusion_2035
    .groupby("업종명")
    .size()
    .reset_index(name="착시성장사례수")
    .sort_values("착시성장사례수", ascending=False)
)

illusion_industry

In [ ]:
illusion_2040 = current_growth[
    current_growth["2040 순인구효과"] < 0
].copy()

print("현재 성장 사례 수 :", len(current_growth))
print("2035 착시성장 사례 수 :", len(illusion_2035))
print("2040 착시성장 사례 수 :", len(illusion_2040))

print(
    "2035 착시성장 비율 :",
    round(len(illusion_2035) / len(current_growth) * 100, 1),
    "%"
)

print(
    "2040 착시성장 비율 :",
    round(len(illusion_2040) / len(current_growth) * 100, 1),
    "%"
)

In [ ]:
illusion_industry_2035 = (
    illusion_2035
    .groupby("업종명")
    .size()
    .reset_index(name="2035착시성장사례수")
)

illusion_industry_2040 = (
    illusion_2040
    .groupby("업종명")
    .size()
    .reset_index(name="2040착시성장사례수")
)

illusion_compare = (
    illusion_industry_2035
    .merge(illusion_industry_2040, on="업종명", how="outer")
    .fillna(0)
)

illusion_compare[["2035착시성장사례수", "2040착시성장사례수"]] = (
    illusion_compare[["2035착시성장사례수", "2040착시성장사례수"]]
    .astype(int)
)

illusion_compare = illusion_compare.sort_values(
    ["2040착시성장사례수", "2035착시성장사례수"],
    ascending=False
)

illusion_compare

In [ ]:
summary_table = pd.DataFrame({
    "구분": ["현재 성장 사례", "2035 착시성장", "2040 착시성장"],
    "사례수": [
        len(current_growth),
        len(illusion_2035),
        len(illusion_2040)
    ]
})

summary_table["비율(%)"] = [
    100.0,
    round(len(illusion_2035) / len(current_growth) * 100, 1),
    round(len(illusion_2040) / len(current_growth) * 100, 1)
]

summary_table

In [ ]:
# 2035와 2040 사이에 착시성장 여부가 바뀐 사례 찾기
changed_cases = current_growth[
    (current_growth["2035 순인구효과"] < 0)
    !=
    (current_growth["2040 순인구효과"] < 0)
].copy()

print("2035 ↔ 2040 상태가 바뀐 사례 수 :", len(changed_cases))

changed_cases[
    [
        "시군구",
        "업종명",
        "2035 순인구효과",
        "2040 순인구효과"
    ]
]

In [ ]:
current_growth["2035→2040 변화"] = (
    current_growth["2040 순인구효과"]
    - current_growth["2035 순인구효과"]
)

print(
    "2040에 더 악화된 사례 수 :",
    (current_growth["2035→2040 변화"] < 0).sum()
)

print(
    "2040에 개선된 사례 수 :",
    (current_growth["2035→2040 변화"] > 0).sum()
)

print(
    "변화 없음 :",
    (current_growth["2035→2040 변화"] == 0).sum()
)

In [ ]:
illusion_persistent = current_growth[
    (current_growth["2035 순인구효과"] < 0) &
    (current_growth["2040 순인구효과"] < 0)
].copy()

illusion_persistent["위험심화폭"] = (
    illusion_persistent["2040 순인구효과"]
    - illusion_persistent["2035 순인구효과"]
)

print("지속 착시성장 사례 :", len(illusion_persistent))
print(
    "그중 2040에 더 악화 :",
    (illusion_persistent["위험심화폭"] < 0).sum()
)

illusion_persistent[
    [
        "시군구",
        "업종명",
        "2035 순인구효과",
        "2040 순인구효과",
        "위험심화폭"
    ]
].sort_values("위험심화폭").head(10)

In [ ]:
current_growth = seoul_result[
    seoul_result["월평균증감률"] > 0
]

In [ ]:
# 기존에 정의한 엄격한 성장 기준 사용
current_growth_strict = seoul_result[
    seoul_result["성장여부"] == True
].copy()

illusion_2035_strict = current_growth_strict[
    current_growth_strict["2035 순인구효과"] < 0
].copy()

illusion_2040_strict = current_growth_strict[
    current_growth_strict["2040 순인구효과"] < 0
].copy()

print("현재 성장 사례 :", len(current_growth_strict))

print(
    "2035 착시성장 :",
    len(illusion_2035_strict),
    "/",
    len(current_growth_strict),
    "=",
    round(
        len(illusion_2035_strict) /
        len(current_growth_strict) * 100, 1
    ),
    "%"
)

print(
    "2040 착시성장 :",
    len(illusion_2040_strict),
    "/",
    len(current_growth_strict),
    "=",
    round(
        len(illusion_2040_strict) /
        len(current_growth_strict) * 100, 1
    ),
    "%"
)

In [ ]:
strict_growth_check = current_growth_strict[
    [
        "시군구",
        "업종명",
        "연령대",
        "월평균증감률",
        "1월대비6월증감률",
        "상승개월수",
        "2035 순인구효과",
        "2040 순인구효과"
    ]
].sort_values("2035 순인구효과")

strict_growth_check

In [ ]:
current_growth_strict["2035→2040 변화"] = (
    current_growth_strict["2040 순인구효과"]
    - current_growth_strict["2035 순인구효과"]
)

print(
    "2040에 더 악화된 사례:",
    (current_growth_strict["2035→2040 변화"] < 0).sum(),
    "/",
    len(current_growth_strict)
)

print(
    "2040에 개선된 사례:",
    (current_growth_strict["2035→2040 변화"] > 0).sum()
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 그래프용 데이터
stress_14 = current_growth_strict.copy()

# 지역 + 업종을 하나의 라벨로
stress_14["사례"] = (
    stress_14["시군구"]
    + " · "
    + stress_14["업종명"]
)

# 2035 위험이 큰 순서로 정렬
stress_14 = stress_14.sort_values(
    "2035 순인구효과",
    ascending=True
)

y = np.arange(len(stress_14))
height = 0.35

plt.figure(figsize=(10, 8))

plt.barh(
    y + height/2,
    stress_14["2035 순인구효과"],
    height,
    label="2035"
)

plt.barh(
    y - height/2,
    stress_14["2040 순인구효과"],
    height,
    label="2040"
)

plt.axvline(0, linewidth=1)

plt.yticks(
    y,
    stress_14["사례"]
)

plt.xlabel("미래 소비기반 스트레스(순인구효과)")
plt.title(
    "미래 소비기반 스트레스 테스트\n"
    "현재 성장 사례 14개 모두 2040년까지 추가 악화"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    "../output/stress_test_14_cases.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
stress_table = current_growth_strict.copy()

# 2035 → 2040 악화 정도
stress_table["2040 추가악화폭"] = (
    stress_table["2040 순인구효과"]
    - stress_table["2035 순인구효과"]
).round(2)

# 보기 좋게 반올림
stress_table["월평균증감률"] = stress_table["월평균증감률"].round(1)
stress_table["1월대비6월증감률"] = stress_table["1월대비6월증감률"].round(1)
stress_table["2035 순인구효과"] = stress_table["2035 순인구효과"].round(2)
stress_table["2040 순인구효과"] = stress_table["2040 순인구효과"].round(2)

# PDF용 표
stress_table_pdf = (
    stress_table[
        [
            "시군구",
            "업종명",
            "연령대",
            "월평균증감률",
            "1월대비6월증감률",
            "2035 순인구효과",
            "2040 순인구효과",
            "2040 추가악화폭"
        ]
    ]
    .rename(columns={
        "연령대": "현재 핵심연령대",
        "월평균증감률": "월평균 성장률",
        "1월대비6월증감률": "1→6월 증감률",
        "2035 순인구효과": "2035 스트레스",
        "2040 순인구효과": "2040 스트레스"
    })
    .sort_values("2040 스트레스")
    .reset_index(drop=True)
)

stress_table_pdf

In [ ]:
# 2040 위험이 큰 순서
rep_pool = current_growth_strict.sort_values(
    "2040 순인구효과"
).copy()

selected_rows = []
used_industries = set()
used_regions = set()

for _, row in rep_pool.iterrows():

    # 업종과 지역이 모두 겹치지 않는 사례 우선
    if (
        row["업종명"] not in used_industries
        and row["시군구"] not in used_regions
    ):
        selected_rows.append(row)

        used_industries.add(row["업종명"])
        used_regions.add(row["시군구"])

    if len(selected_rows) == 3:
        break

representative_3 = pd.DataFrame(selected_rows)

representative_3[
    [
        "시군구",
        "업종명",
        "연령대",
        "월평균증감률",
        "1월대비6월증감률",
        "2035 순인구효과",
        "2040 순인구효과"
    ]
]

In [ ]:
print(owner_strategy_v2.columns.tolist())

In [ ]:
rep_solution = owner_strategy_v2[
    owner_strategy_v2.apply(
        lambda row:
        (
            (row["시군구"] == "성동구") &
            (row["업종명"] == "서양음식")
        )
        or
        (
            (row["시군구"] == "도봉구") &
            (row["업종명"] == "편의점")
        )
        or
        (
            (row["시군구"] == "성북구") &
            (row["업종명"] == "일반한식")
        ),
        axis=1
    )
].copy()

rep_solution

In [ ]:
# 대표 3개 현재 성장정보 + 전략정보 결합
rep_3_detail = representative_3[
    [
        "시군구",
        "업종명",
        "연령대",
        "월평균증감률",
        "1월대비6월증감률",
        "2035 순인구효과",
        "2040 순인구효과"
    ]
].copy()

rep_3_detail = rep_3_detail.rename(
    columns={"연령대": "현재핵심연령대"}
)

# owner_strategy_v2에서 필요한 전략 컬럼만 붙이기
strategy_cols = [
    "시군구",
    "업종명",
    "방어형고객",
    "방어형유형",
    "방어전략상태",
    "방어개선잠재력",
    "전환후_2040순인구효과",
    "2040위험완화율"
]

rep_3_detail = rep_3_detail.merge(
    owner_strategy_v2[strategy_cols],
    on=["시군구", "업종명"],
    how="left"
)

# 2035 전환 후 효과 계산
rep_3_detail["전환후_2035순인구효과"] = (
    rep_3_detail["2035 순인구효과"]
    + rep_3_detail["방어개선잠재력"]
).round(2)

# 보기 좋게 반올림
for col in [
    "월평균증감률",
    "1월대비6월증감률",
    "2035 순인구효과",
    "2040 순인구효과",
    "방어개선잠재력",
    "전환후_2040순인구효과",
    "2040위험완화율"
]:
    rep_3_detail[col] = rep_3_detail[col].round(2)

rep_3_detail

In [ ]:
def make_case_message(row):
    if row["방어전략상태"] == "방어형 후보 없음":
        return "구조적 취약형"
    
    elif row["방어형유형"] == "성장·방어형":
        return "성장고객 전환형"
    
    elif row["방어형유형"] == "상대안정형":
        return "상대안정 고객 전환형"
    
    else:
        return "전환전략 검토형"


rep_3_detail["대표사례유형"] = rep_3_detail.apply(
    make_case_message,
    axis=1
)

case_summary = rep_3_detail[
    [
        "시군구",
        "업종명",
        "현재핵심연령대",
        "대표사례유형",
        "방어형고객",
        "2035 순인구효과",
        "2040 순인구효과",
        "전환후_2040순인구효과",
        "2040위험완화율"
    ]
]

case_summary

In [ ]:
case_summary_pdf = rep_3_detail[
    [
        "시군구",
        "업종명",
        "현재핵심연령대",
        "대표사례유형",
        "방어형고객",
        "2040 순인구효과",
        "전환후_2040순인구효과",
        "2040위험완화율"
    ]
].copy()

case_summary_pdf = case_summary_pdf.rename(
    columns={
        "시군구": "지역",
        "업종명": "업종",
        "현재핵심연령대": "현재 핵심고객",
        "대표사례유형": "진단",
        "방어형고객": "방어형 고객",
        "2040 순인구효과": "전환 전",
        "전환후_2040순인구효과": "전환 후",
        "2040위험완화율": "위험완화율(%)"
    }
)

# 후보가 없는 경우 보기 좋게 표시
case_summary_pdf["방어형 고객"] = (
    case_summary_pdf["방어형 고객"]
    .fillna("후보 없음")
)

case_summary_pdf["전환 후"] = (
    case_summary_pdf["전환 후"]
    .apply(lambda x: "-" if pd.isna(x) else round(x, 2))
)

case_summary_pdf["위험완화율(%)"] = (
    case_summary_pdf["위험완화율(%)"]
    .apply(lambda x: "-" if pd.isna(x) else round(x, 1))
)

case_summary_pdf

In [ ]:
# 엄격한 성장 사례 14개만 추출
strict_keys = (
    current_growth_strict[
        ["시군구", "업종명"]
    ]
    .drop_duplicates()
)

# 14개 사례에 기존 점주 전략 결과 연결
strict_owner = strict_keys.merge(
    owner_strategy_v2,
    on=["시군구", "업종명"],
    how="left"
)

print("전체 착시성장 사례 :", len(strict_owner))

print("\n[방어전략 상태]")
print(
    strict_owner["방어전략상태"]
    .value_counts(dropna=False)
)

print("\n장기유효 사례 :",
      (strict_owner["방어전략상태"] == "장기유효").sum())

print("장기재검토 사례 :",
      (strict_owner["방어전략상태"] == "장기재검토").sum())

print("방어형 후보 없음 :",
      (strict_owner["방어전략상태"] == "방어형 후보 없음").sum())

In [ ]:
valid_defense = strict_owner[
    strict_owner["방어전략상태"] == "장기유효"
].copy()

print("장기유효 방어전략 수 :", len(valid_defense))

print(
    "평균 2040 위험완화율 :",
    round(valid_defense["2040위험완화율"].mean(), 1),
    "%"
)

print(
    "최대 2040 위험완화율 :",
    round(valid_defense["2040위험완화율"].max(), 1),
    "%"
)

print(
    "최소 2040 위험완화율 :",
    round(valid_defense["2040위험완화율"].min(), 1),
    "%"
)

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

status_count = (
    strict_owner["방어전략상태"]
    .value_counts()
    .reindex(
        ["장기유효", "장기재검토", "방어형 후보 없음"],
        fill_value=0
    )
)

plt.figure(figsize=(7, 5))

bars = plt.bar(
    status_count.index,
    status_count.values
)

plt.ylabel("사례 수")
plt.title(
    "착시성장 14건의 Next Customer 처방 결과\n"
    "11건에서 2040년까지 장기 유효한 방어전략 확인"
)

plt.ylim(0, 14)

# 막대 위 숫자 표시
for bar, value in zip(bars, status_count.values):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.25,
        str(value),
        ha="center",
        fontsize=12
    )

plt.tight_layout()

plt.savefig(
    "../output/next_customer_status_14_cases.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
stress_table_pdf.to_csv(
    "../output/stress_test_14_cases_table.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
case_summary_pdf.to_csv(
    "../output/representative_3_cases_table.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 성동구 서양음식 연령대별 데이터
case_seongdong = seoul_age_stress[
    (seoul_age_stress["시군구"] == "성동구") &
    (seoul_age_stress["업종명"] == "서양음식")
].copy()

# 연령대 순서
age_order = [
    "20대이하",
    "20대",
    "30대",
    "40대",
    "50대",
    "60대이상"
]

case_seongdong["연령대"] = pd.Categorical(
    case_seongdong["연령대"],
    categories=age_order,
    ordered=True
)

case_seongdong = case_seongdong.sort_values("연령대")

# 확인
case_seongdong[
    [
        "연령대",
        "이용금액비중",
        "2035 증감률",
        "2040 증감률",
        "2040 순인구효과"
    ]
]

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

plt.figure(figsize=(8, 5))

plt.bar(
    case_seongdong["연령대"],
    case_seongdong["2040 순인구효과"]
)

plt.axhline(0, linewidth=1)

plt.xlabel("연령대")
plt.ylabel("2040 순인구효과")

plt.title(
    "성동구 서양음식\n"
    "연령대별 2040 미래 소비기반 영향"
)

plt.tight_layout()

plt.savefig(
    "../output/seongdong_western_age_2040.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 성동구 서양음식 전략 결과
seongdong_strategy = owner_strategy_v2[
    (owner_strategy_v2["시군구"] == "성동구") &
    (owner_strategy_v2["업종명"] == "서양음식")
].iloc[0]

# 전환 전
before = [
    seongdong_strategy["2035 순인구효과"],
    seongdong_strategy["2040 순인구효과"]
]

# 2035 전환 후
after_2035 = (
    seongdong_strategy["2035 순인구효과"]
    + seongdong_strategy["방어개선잠재력"]
)

# 2040 전환 후
after_2040 = seongdong_strategy["전환후_2040순인구효과"]

after = [
    after_2035,
    after_2040
]

years = ["2035", "2040"]

x = np.arange(len(years))
width = 0.35

plt.figure(figsize=(8, 5))

bars1 = plt.bar(
    x - width/2,
    before,
    width,
    label="전환 전"
)

bars2 = plt.bar(
    x + width/2,
    after,
    width,
    label="전환 후"
)

plt.axhline(0, linewidth=1)

plt.xticks(x, years)
plt.ylabel("미래 소비기반 스트레스")

plt.title(
    "성동구 서양음식\n"
    "30대 → 60대 이상 고객전환 What-if"
)

plt.legend()

# 막대에 값 표시
for bars in [bars1, bars2]:
    for bar in bars:
        value = bar.get_height()

        plt.text(
            bar.get_x() + bar.get_width()/2,
            value - 0.4,
            f"{value:.2f}",
            ha="center",
            va="top",
            fontsize=10
        )

plt.tight_layout()

plt.savefig(
    "../output/seongdong_western_transition_effect.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 도봉구 편의점 연령대별 데이터
case_dobong = seoul_age_stress[
    (seoul_age_stress["시군구"] == "도봉구") &
    (seoul_age_stress["업종명"] == "편의점")
].copy()

# 연령대 순서
age_order = [
    "20대이하",
    "20대",
    "30대",
    "40대",
    "50대",
    "60대이상"
]

case_dobong["연령대"] = pd.Categorical(
    case_dobong["연령대"],
    categories=age_order,
    ordered=True
)

case_dobong = case_dobong.sort_values("연령대")

# 그래프
plt.figure(figsize=(8, 5))

plt.bar(
    case_dobong["연령대"],
    case_dobong["2040 순인구효과"]
)

plt.axhline(0, linewidth=1)

plt.xlabel("연령대")
plt.ylabel("2040 순인구효과")

plt.title(
    "도봉구 편의점\n"
    "연령대별 2040 미래 소비기반 영향"
)

plt.tight_layout()

plt.savefig(
    "../output/dobong_convenience_age_2040.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 도봉구 편의점 전략 결과
dobong_strategy = owner_strategy_v2[
    (owner_strategy_v2["시군구"] == "도봉구") &
    (owner_strategy_v2["업종명"] == "편의점")
].iloc[0]

before = [
    dobong_strategy["2035 순인구효과"],
    dobong_strategy["2040 순인구효과"]
]

after_2035 = (
    dobong_strategy["2035 순인구효과"]
    + dobong_strategy["방어개선잠재력"]
)

after_2040 = dobong_strategy["전환후_2040순인구효과"]

after = [
    after_2035,
    after_2040
]

years = ["2035", "2040"]

x = np.arange(len(years))
width = 0.35

plt.figure(figsize=(8, 5))

bars1 = plt.bar(
    x - width/2,
    before,
    width,
    label="전환 전"
)

bars2 = plt.bar(
    x + width/2,
    after,
    width,
    label="전환 후"
)

plt.axhline(0, linewidth=1)

plt.xticks(x, years)
plt.ylabel("미래 소비기반 스트레스")

plt.title(
    "도봉구 편의점\n"
    "50대 → 30대 고객전환 What-if"
)

plt.legend()

# 값 표시
for bars in [bars1, bars2]:
    for bar in bars:
        value = bar.get_height()

        plt.text(
            bar.get_x() + bar.get_width()/2,
            value - 0.35,
            f"{value:.2f}",
            ha="center",
            va="top",
            fontsize=10
        )

plt.tight_layout()

plt.savefig(
    "../output/dobong_convenience_transition_effect.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 성북구 일반한식 연령대별 데이터
case_seongbuk = seoul_age_stress[
    (seoul_age_stress["시군구"] == "성북구") &
    (seoul_age_stress["업종명"] == "일반한식")
].copy()

# 연령대 순서
age_order = [
    "20대이하",
    "20대",
    "30대",
    "40대",
    "50대",
    "60대이상"
]

case_seongbuk["연령대"] = pd.Categorical(
    case_seongbuk["연령대"],
    categories=age_order,
    ordered=True
)

case_seongbuk = case_seongbuk.sort_values("연령대")

# 그래프
plt.figure(figsize=(8, 5))

plt.bar(
    case_seongbuk["연령대"],
    case_seongbuk["2040 순인구효과"]
)

plt.axhline(0, linewidth=1)

plt.xlabel("연령대")
plt.ylabel("2040 순인구효과")

plt.title(
    "성북구 일반한식\n"
    "연령대별 2040 미래 소비기반 영향"
)

plt.tight_layout()

plt.savefig(
    "../output/seongbuk_korean_age_2040.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 성북구 일반한식 전략 결과
seongbuk_strategy = owner_strategy_v2[
    (owner_strategy_v2["시군구"] == "성북구") &
    (owner_strategy_v2["업종명"] == "일반한식")
].iloc[0]

years = ["2035", "2040"]

stress_values = [
    seongbuk_strategy["2035 순인구효과"],
    seongbuk_strategy["2040 순인구효과"]
]

plt.figure(figsize=(7, 5))

bars = plt.bar(
    years,
    stress_values
)

plt.axhline(0, linewidth=1)

plt.ylabel("미래 소비기반 스트레스")

plt.title(
    "성북구 일반한식\n"
    "방어형 고객 후보 없음 · 미래 위험 지속"
)

# 값 표시
for bar in bars:
    value = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width()/2,
        value - 0.15,
        f"{value:.2f}",
        ha="center",
        va="top",
        fontsize=11
    )

plt.tight_layout()

plt.savefig(
    "../output/seongbuk_korean_structural_risk.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 성북구 일반한식 전략 결과
seongbuk_strategy = owner_strategy_v2[
    (owner_strategy_v2["시군구"] == "성북구") &
    (owner_strategy_v2["업종명"] == "일반한식")
].iloc[0]

years = ["2035", "2040"]

stress_values = [
    seongbuk_strategy["2035 순인구효과"],
    seongbuk_strategy["2040 순인구효과"]
]

plt.figure(figsize=(7, 5))

bars = plt.bar(
    years,
    stress_values
)

plt.axhline(0, linewidth=1)

plt.ylabel("미래 소비기반 스트레스")

plt.title(
    "성북구 일반한식\n"
    "방어형 고객 후보 없음 · 미래 위험 지속"
)

# 값 표시
for bar in bars:
    value = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.15,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=11
    )

plt.tight_layout()

plt.savefig(
    "../output/seongbuk_korean_structural_risk.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# 1. 전체 개인 소비 데이터
n_person = len(bc_clean)

# 2. 서울 개인 소비 데이터
n_seoul_rows = len(
    bc_clean[bc_clean["시도"] == "서울특별시"]
)

# 3. 서울 지역×업종 전체 조합 수
seoul_all_cases = (
    bc_profile[
        bc_profile["시도"] == "서울특별시"
    ][["시군구", "업종명"]]
    .drop_duplicates()
)

# 4. 6개월 완전관측 사례
seoul_complete_cases = (
    bc_profile[
        (bc_profile["시도"] == "서울특별시") &
        (bc_profile["6개월완전"] == True)
    ][["시군구", "업종명"]]
    .drop_duplicates()
)

# 5. 규모 상위 25%까지 통과한 분석 대상
seoul_analysis_cases = (
    analysis_base[
        analysis_base["시도"] == "서울특별시"
    ][["시군구", "업종명"]]
    .drop_duplicates()
)

# 6. 엄격한 성장 조건까지 통과
strict_cases = (
    current_growth_strict[
        ["시군구", "업종명"]
    ]
    .drop_duplicates()
)

print("개인 소비 행 수 :", n_person)
print("서울 개인 소비 행 수 :", n_seoul_rows)
print("서울 지역×업종 전체 :", len(seoul_all_cases))
print("6개월 완전관측 :", len(seoul_complete_cases))
print("규모 상위 25% 분석대상 :", len(seoul_analysis_cases))
print("엄격한 성장조건 충족 :", len(strict_cases))

In [ ]:
# ------------------------------------------------
# 1. 서울 전체 지역×업종에 대해 미래 스트레스 다시 계산
# ------------------------------------------------

seoul_age_all = bc_age[
    bc_age["시도"] == "서울특별시"
].copy()

seoul_age_all = seoul_age_all.merge(
    future_seoul[
        [
            "시군구",
            "연령대",
            "2035 증감률",
            "2040 증감률"
        ]
    ],
    on=["시군구", "연령대"],
    how="left"
)

share = seoul_age_all["이용금액비중"] / 100

seoul_age_all["2035 순인구효과"] = (
    share * seoul_age_all["2035 증감률"]
)

seoul_age_all["2040 순인구효과"] = (
    share * seoul_age_all["2040 증감률"]
)

seoul_stress_all = (
    seoul_age_all
    .groupby(
        ["시도", "시군구", "업종코드", "업종명"],
        as_index=False
    )[
        ["2035 순인구효과", "2040 순인구효과"]
    ]
    .sum()
)

# 서울 전체 프로필 + 스트레스
sensitivity_base = (
    bc_profile[
        bc_profile["시도"] == "서울특별시"
    ]
    .merge(
        seoul_stress_all,
        on=["시도", "시군구", "업종코드", "업종명"],
        how="left"
    )
)


# ------------------------------------------------
# 2. 규모 기준을 바꾸며 검증
# ------------------------------------------------

results = []

for top_pct in [20, 25, 30]:

    # 상위 20% = 80분위 이상
    cutoff = bc_profile["업종총이용금액"].quantile(
        1 - top_pct / 100
    )

    test = sensitivity_base[
        (sensitivity_base["6개월완전"] == True) &
        (sensitivity_base["업종총이용금액"] >= cutoff) &
        (sensitivity_base["월평균증감률"] > 0) &
        (sensitivity_base["1월대비6월증감률"] > 0) &
        (sensitivity_base["상승개월수"] >= 3)
    ].copy()

    n = len(test)

    risk_2035 = (test["2035 순인구효과"] < 0).sum()

    worse_2040 = (
        test["2040 순인구효과"]
        <
        test["2035 순인구효과"]
    ).sum()

    results.append({
        "규모기준": f"상위 {top_pct}%",
        "현재성장사례": n,
        "2035 위험사례": risk_2035,
        "2035 위험비율(%)":
            round(risk_2035 / n * 100, 1) if n > 0 else None,
        "2040 추가악화": worse_2040,
        "2040 추가악화비율(%)":
            round(worse_2040 / n * 100, 1) if n > 0 else None
    })

sensitivity_size = pd.DataFrame(results)

sensitivity_size

In [ ]:
# 비교 대상: 서울 + 6개월 완전관측 사례 전체
baseline_cases = sensitivity_base[
    sensitivity_base["6개월완전"] == True
].copy()

# 엄격한 성장조건
baseline_cases["엄격성장"] = (
    (baseline_cases["월평균증감률"] > 0) &
    (baseline_cases["1월대비6월증감률"] > 0) &
    (baseline_cases["상승개월수"] >= 3)
)

# 성장 아닌 사례
growth_cases = baseline_cases[
    baseline_cases["엄격성장"] == True
]

non_growth_cases = baseline_cases[
    baseline_cases["엄격성장"] == False
]

def risk_summary(df, name):
    n = len(df)

    risk_2035 = (df["2035 순인구효과"] < 0).sum()

    worse_2040 = (
        df["2040 순인구효과"]
        < df["2035 순인구효과"]
    ).sum()

    return {
        "구분": name,
        "사례수": n,
        "2035 위험사례": risk_2035,
        "2035 위험비율(%)":
            round(risk_2035 / n * 100, 1),
        "2040 추가악화": worse_2040,
        "2040 추가악화비율(%)":
            round(worse_2040 / n * 100, 1)
    }

baseline_compare = pd.DataFrame([
    risk_summary(baseline_cases, "서울 전체 완전관측"),
    risk_summary(growth_cases, "엄격한 성장"),
    risk_summary(non_growth_cases, "비성장")
])

baseline_compare

In [ ]:
# 기존 분석과 동일한 상위 25% 규모 기준
q75 = bc_profile["업종총이용금액"].quantile(0.75)

same_base = sensitivity_base[
    (sensitivity_base["6개월완전"] == True) &
    (sensitivity_base["업종총이용금액"] >= q75)
].copy()

# 엄격한 성장 조건
same_base["엄격성장"] = (
    (same_base["월평균증감률"] > 0) &
    (same_base["1월대비6월증감률"] > 0) &
    (same_base["상승개월수"] >= 3)
)

growth_same = same_base[
    same_base["엄격성장"] == True
]

nongrowth_same = same_base[
    same_base["엄격성장"] == False
]

def compare_group(df, name):
    n = len(df)

    risk2035 = (df["2035 순인구효과"] < 0).sum()

    worse2040 = (
        df["2040 순인구효과"]
        < df["2035 순인구효과"]
    ).sum()

    return {
        "구분": name,
        "사례수": n,
        "2035 위험사례": risk2035,
        "2035 위험비율(%)":
            round(risk2035 / n * 100, 1),
        "2035 스트레스 중앙값":
            round(df["2035 순인구효과"].median(), 2),
        "2040 추가악화": worse2040,
        "2040 추가악화비율(%)":
            round(worse2040 / n * 100, 1),
        "2040 스트레스 중앙값":
            round(df["2040 순인구효과"].median(), 2)
    }

same_group_compare = pd.DataFrame([
    compare_group(same_base, "상위25% 전체"),
    compare_group(growth_same, "상위25% 현재성장"),
    compare_group(nongrowth_same, "상위25% 비성장")
])

same_group_compare

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

groups = ["현재 성장", "비성장"]
risk_rates = [100.0, 74.0]

plt.figure(figsize=(7, 5))

bars = plt.bar(
    groups,
    risk_rates
)

plt.ylabel("2035 미래 위험 비율(%)")
plt.ylim(0, 110)

plt.title(
    "같은 소비규모에서도 미래 위험은 달랐다\n"
    "이용규모 상위 25% 성장 vs 비성장"
)

for bar, value in zip(bars, risk_rates):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + 2,
        f"{value:.1f}%",
        ha="center",
        fontsize=12
    )

plt.tight_layout()

plt.savefig(
    "../output/growth_vs_nongrowth_risk_2035.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
groups = ["현재 성장", "비성장"]

stress_2035 = [-7.70, -3.89]
stress_2040 = [-11.61, -6.67]

import numpy as np

x = np.arange(len(groups))
width = 0.35

plt.figure(figsize=(8, 5))

bars1 = plt.bar(
    x - width/2,
    stress_2035,
    width,
    label="2035"
)

bars2 = plt.bar(
    x + width/2,
    stress_2040,
    width,
    label="2040"
)

plt.axhline(0, linewidth=1)

plt.xticks(x, groups)
plt.ylabel("미래 소비기반 스트레스 중앙값")

plt.title(
    "현재 성장 사례의 미래 스트레스가 더 컸다\n"
    "이용규모 상위 25% 기준"
)

plt.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        value = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2,
            value + 0.3,
            f"{value:.2f}",
            ha="center",
            va="bottom",
            fontsize=10
        )

plt.tight_layout()

plt.savefig(
    "../output/growth_vs_nongrowth_stress.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from scipy.stats import fisher_exact

# 현재 성장: 위험 14 / 비위험 0
# 비성장: 위험 74 / 비위험 26
risk_table_2035 = [
    [14, 0],
    [74, 26]
]

oddsratio, p_value = fisher_exact(
    risk_table_2035,
    alternative="greater"
)

print("2035 위험비율 Fisher 검정")
print("p-value :", round(p_value, 4))

In [ ]:
%pip install scipy

In [ ]:
from scipy.stats import fisher_exact

# 현재 성장: 위험 14 / 비위험 0
# 비성장: 위험 74 / 비위험 26
risk_table_2035 = [
    [14, 0],
    [74, 26]
]

oddsratio, p_value = fisher_exact(
    risk_table_2035,
    alternative="greater"
)

print("2035 위험비율 Fisher 검정")
print("p-value :", round(p_value, 4))

In [ ]:
import scipy
print(scipy.__version__)

In [ ]:
from scipy.stats import fisher_exact

In [ ]:
from scipy.stats import fisher_exact

# 현재 성장: 위험 14 / 비위험 0
# 비성장: 위험 74 / 비위험 26
risk_table_2035 = [
    [14, 0],
    [74, 26]
]

oddsratio, p_value = fisher_exact(
    risk_table_2035,
    alternative="greater"
)

print("2035 위험비율 Fisher 검정")
print("p-value :", round(p_value, 4))

In [ ]:
from scipy.stats import mannwhitneyu

# 2035
stat_2035, p_2035 = mannwhitneyu(
    growth_same["2035 순인구효과"],
    nongrowth_same["2035 순인구효과"],
    alternative="less"
)

# 2040
stat_2040, p_2040 = mannwhitneyu(
    growth_same["2040 순인구효과"],
    nongrowth_same["2040 순인구효과"],
    alternative="less"
)

print("2035 스트레스 차이 p-value :", round(p_2035, 4))
print("2040 스트레스 차이 p-value :", round(p_2040, 4))

In [ ]:
import pandas as pd

# -----------------------------------------
# 1. 저장해둔 데이터 다시 불러오기
# -----------------------------------------

bc_profile = pd.read_csv(
    "../data/processed/bc_region_industry_profile.csv",
    encoding="utf-8-sig"
)

bc_age = pd.read_csv(
    "../data/processed/bc_age_consumption_structure.csv",
    encoding="utf-8-sig"
)

future_change = pd.read_csv(
    "../data/processed/seoul_future_population_change.csv",
    encoding="utf-8-sig"
)

# 미래인구 연령대 이름 맞추기
future_seoul = future_change.copy()

future_seoul["연령대"] = future_seoul["연령대"].replace({
    "20대미만": "20대이하"
})

future_seoul = future_seoul.rename(
    columns={"자치구별": "시군구"}
)


# -----------------------------------------
# 2. 서울 전체 미래 스트레스 계산
# -----------------------------------------

seoul_age_all = bc_age[
    bc_age["시도"] == "서울특별시"
].copy()

seoul_age_all = seoul_age_all.merge(
    future_seoul[
        [
            "시군구",
            "연령대",
            "2035 증감률",
            "2040 증감률"
        ]
    ],
    on=["시군구", "연령대"],
    how="left"
)

share = seoul_age_all["이용금액비중"] / 100

seoul_age_all["2035 순인구효과"] = (
    share * seoul_age_all["2035 증감률"]
)

seoul_age_all["2040 순인구효과"] = (
    share * seoul_age_all["2040 증감률"]
)

seoul_stress_all = (
    seoul_age_all
    .groupby(
        ["시도", "시군구", "업종코드", "업종명"],
        as_index=False
    )[
        ["2035 순인구효과", "2040 순인구효과"]
    ]
    .sum()
)

sensitivity_base = (
    bc_profile[
        bc_profile["시도"] == "서울특별시"
    ]
    .merge(
        seoul_stress_all,
        on=["시도", "시군구", "업종코드", "업종명"],
        how="left"
    )
)


# -----------------------------------------
# 3. 기존과 동일하게 상위 25% 기준
# -----------------------------------------

q75 = bc_profile["업종총이용금액"].quantile(0.75)

same_base = sensitivity_base[
    (sensitivity_base["6개월완전"] == True) &
    (sensitivity_base["업종총이용금액"] >= q75)
].copy()

# 엄격한 성장조건
same_base["엄격성장"] = (
    (same_base["월평균증감률"] > 0) &
    (same_base["1월대비6월증감률"] > 0) &
    (same_base["상승개월수"] >= 3)
)

growth_same = same_base[
    same_base["엄격성장"] == True
].copy()

nongrowth_same = same_base[
    same_base["엄격성장"] == False
].copy()

print("상위 25% 전체 :", len(same_base))
print("현재 성장 :", len(growth_same))
print("비성장 :", len(nongrowth_same))

In [ ]:
from scipy.stats import mannwhitneyu

# 2035
stat_2035, p_2035 = mannwhitneyu(
    growth_same["2035 순인구효과"],
    nongrowth_same["2035 순인구효과"],
    alternative="less"
)

# 2040
stat_2040, p_2040 = mannwhitneyu(
    growth_same["2040 순인구효과"],
    nongrowth_same["2040 순인구효과"],
    alternative="less"
)

print(
    "2035 스트레스 차이 p-value :",
    round(p_2035, 4)
)

print(
    "2040 스트레스 차이 p-value :",
    round(p_2040, 4)
)

In [ ]:
validation_summary = pd.DataFrame({
    "구분": ["현재 성장", "비성장"],
    "사례수": [14, 100],
    "2035 위험비율(%)": [100.0, 74.0],
    "2035 스트레스 중앙값": [-7.70, -3.89],
    "2040 추가악화비율(%)": [100.0, 84.0],
    "2040 스트레스 중앙값": [-11.61, -6.67]
})

validation_summary.to_csv(
    "../output/growth_vs_nongrowth_validation.csv",
    index=False,
    encoding="utf-8-sig"
)

validation_summary

In [ ]:
stat_test_summary = pd.DataFrame({
    "검정": [
        "2035 위험비율 Fisher exact",
        "2035 스트레스 Mann-Whitney U",
        "2040 스트레스 Mann-Whitney U"
    ],
    "p-value": [
        0.0206,
        0.0008,
        0.0006
    ]
})

stat_test_summary.to_csv(
    "../output/statistical_validation.csv",
    index=False,
    encoding="utf-8-sig"
)

stat_test_summary

In [ ]:
sensitivity_size.to_csv(
    "../output/size_threshold_sensitivity.csv",
    index=False,
    encoding="utf-8-sig"
)

sensitivity_size

In [ ]:
results = []

for top_pct in [20, 25, 30]:

    cutoff = bc_profile["업종총이용금액"].quantile(
        1 - top_pct / 100
    )

    test = sensitivity_base[
        (sensitivity_base["6개월완전"] == True) &
        (sensitivity_base["업종총이용금액"] >= cutoff) &
        (sensitivity_base["월평균증감률"] > 0) &
        (sensitivity_base["1월대비6월증감률"] > 0) &
        (sensitivity_base["상승개월수"] >= 3)
    ].copy()

    n = len(test)

    risk_2035 = (
        test["2035 순인구효과"] < 0
    ).sum()

    worse_2040 = (
        test["2040 순인구효과"]
        < test["2035 순인구효과"]
    ).sum()

    results.append({
        "규모기준": f"상위 {top_pct}%",
        "현재성장사례": n,
        "2035 위험사례": risk_2035,
        "2035 위험비율(%)":
            round(risk_2035 / n * 100, 1),
        "2040 추가악화": worse_2040,
        "2040 추가악화비율(%)":
            round(worse_2040 / n * 100, 1)
    })

sensitivity_size = pd.DataFrame(results)

# 저장
sensitivity_size.to_csv(
    "../output/size_threshold_sensitivity.csv",
    index=False,
    encoding="utf-8-sig"
)

sensitivity_size

In [ ]:
import pandas as pd

bc_age = pd.read_csv(
    "../data/processed/bc_age_consumption_structure.csv",
    encoding="utf-8-sig"
)

bc_profile = pd.read_csv(
    "../data/processed/bc_region_industry_profile.csv",
    encoding="utf-8-sig"
)

busan_future = pd.read_csv(
    "../data/processed/busan_future_population_change.csv",
    encoding="utf-8-sig"
)

print(bc_age.shape)
print(bc_profile.shape)
print(busan_future.shape)

In [ ]:
# 부산 BC 연령별 소비구조
busan_age_stress = bc_age[
    bc_age["시도"] == "부산광역시"
].copy()

# 부산 미래인구 결합
busan_age_stress = busan_age_stress.merge(
    busan_future[
        [
            "시군구",
            "연령대",
            "2035 증감률",
            "2040 증감률"
        ]
    ],
    on=["시군구", "연령대"],
    how="left"
)

print(
    "미래인구 결측 행:",
    busan_age_stress["2035 증감률"].isna().sum()
)

In [ ]:
share = busan_age_stress["이용금액비중"] / 100

busan_age_stress["2035 순인구효과"] = (
    share * busan_age_stress["2035 증감률"]
)

busan_age_stress["2040 순인구효과"] = (
    share * busan_age_stress["2040 증감률"]
)

busan_stress_summary = (
    busan_age_stress
    .groupby(
        ["시도", "시군구", "업종코드", "업종명"],
        as_index=False
    )[
        ["2035 순인구효과", "2040 순인구효과"]
    ]
    .sum()
)

busan_stress_summary.head()

In [ ]:
busan_result = (
    bc_profile[
        bc_profile["시도"] == "부산광역시"
    ]
    .merge(
        busan_stress_summary,
        on=["시도", "시군구", "업종코드", "업종명"],
        how="left"
    )
)

print("부산 지역×업종 전체 :", len(busan_result))

busan_result.head()

In [ ]:
# 서울에서 사용한 것과 동일한 기준
q75 = bc_profile["업종총이용금액"].quantile(0.75)

busan_same_base = busan_result[
    (busan_result["6개월완전"] == True) &
    (busan_result["업종총이용금액"] >= q75)
].copy()

# 엄격한 성장조건
busan_same_base["엄격성장"] = (
    (busan_same_base["월평균증감률"] > 0) &
    (busan_same_base["1월대비6월증감률"] > 0) &
    (busan_same_base["상승개월수"] >= 3)
)

busan_growth = busan_same_base[
    busan_same_base["엄격성장"] == True
].copy()

busan_nongrowth = busan_same_base[
    busan_same_base["엄격성장"] == False
].copy()

print("부산 상위25% 전체 :", len(busan_same_base))
print("부산 현재 성장 :", len(busan_growth))
print("부산 비성장 :", len(busan_nongrowth))

In [ ]:
def city_group_summary(df, name):
    n = len(df)

    risk_2035 = (
        df["2035 순인구효과"] < 0
    ).sum()

    worse_2040 = (
        df["2040 순인구효과"]
        < df["2035 순인구효과"]
    ).sum()

    return {
        "구분": name,
        "사례수": n,

        "2035 위험사례": risk_2035,

        "2035 위험비율(%)":
            round(risk_2035 / n * 100, 1)
            if n > 0 else None,

        "2035 스트레스 중앙값":
            round(df["2035 순인구효과"].median(), 2)
            if n > 0 else None,

        "2040 추가악화": worse_2040,

        "2040 추가악화비율(%)":
            round(worse_2040 / n * 100, 1)
            if n > 0 else None,

        "2040 스트레스 중앙값":
            round(df["2040 순인구효과"].median(), 2)
            if n > 0 else None
    }


busan_compare = pd.DataFrame([
    city_group_summary(
        busan_same_base,
        "부산 상위25% 전체"
    ),
    city_group_summary(
        busan_growth,
        "부산 현재성장"
    ),
    city_group_summary(
        busan_nongrowth,
        "부산 비성장"
    )
])

busan_compare

In [ ]:
from scipy.stats import mannwhitneyu

# 2035
stat_2035_busan, p_2035_busan = mannwhitneyu(
    busan_growth["2035 순인구효과"],
    busan_nongrowth["2035 순인구효과"],
    alternative="less"
)

# 2040
stat_2040_busan, p_2040_busan = mannwhitneyu(
    busan_growth["2040 순인구효과"],
    busan_nongrowth["2040 순인구효과"],
    alternative="less"
)

print(
    "부산 2035 스트레스 차이 p-value :",
    round(p_2035_busan, 4)
)

print(
    "부산 2040 스트레스 차이 p-value :",
    round(p_2040_busan, 4)
)

In [ ]:
bc_gyeonggi_regions = (
    bc_clean[
        bc_clean["시도"] == "경기도"
    ]["시군구"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("BC 경기도 지역 수 :", len(bc_gyeonggi_regions))
print(bc_gyeonggi_regions)

In [ ]:
import pandas as pd

bc_clean = pd.read_csv(
    "../data/processed/bc_card_clean_202601_202606.csv",
    encoding="utf-8-sig"
)

print(bc_clean.shape)

In [ ]:
bc_gyeonggi_regions = (
    bc_clean[
        bc_clean["시도"] == "경기도"
    ]["시군구"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("BC 경기도 지역 수 :", len(bc_gyeonggi_regions))
print(bc_gyeonggi_regions)

In [ ]:
# 경기도 데이터만 복사
bc_gyeonggi = bc_clean[
    bc_clean["시도"] == "경기도"
].copy()

# 일반구가 있는 도시를 시 단위로 통합
def to_parent_city(name):
    parts = str(name).split()

    # 예: 고양시 덕양구 -> 고양시
    #     수원시 장안구 -> 수원시
    if len(parts) >= 2 and parts[0].endswith("시"):
        return parts[0]

    # 예: 가평군, 과천시 -> 그대로
    return name


bc_gyeonggi["시군"] = (
    bc_gyeonggi["시군구"]
    .apply(to_parent_city)
)

print("통합 전 지역 수 :", bc_gyeonggi["시군구"].nunique())
print("통합 후 지역 수 :", bc_gyeonggi["시군"].nunique())

print(
    sorted(
        bc_gyeonggi["시군"].drop_duplicates().tolist()
    )
)

In [ ]:
# 경기도 BC 데이터를 31개 시·군 기준으로 다시 집계
gyeonggi_bc = (
    bc_gyeonggi
    .groupby(
        [
            "기준년월",
            "시군",
            "연령대",
            "업종코드",
            "업종명"
        ],
        as_index=False
    )[
        ["이용금액", "이용건수"]
    ]
    .sum()
)

print("경기도 시군 수 :", gyeonggi_bc["시군"].nunique())
print("업종 수 :", gyeonggi_bc["업종명"].nunique())

gyeonggi_bc.head()

In [ ]:
gyeonggi_age = (
    gyeonggi_bc
    .groupby(
        ["시군", "업종코드", "업종명", "연령대"],
        as_index=False
    )[
        ["이용금액", "이용건수"]
    ]
    .sum()
)

# 시군×업종 전체 이용금액
gyeonggi_age["업종총이용금액"] = (
    gyeonggi_age
    .groupby(["시군", "업종코드"])["이용금액"]
    .transform("sum")
)

# 연령별 이용금액 비중
gyeonggi_age["이용금액비중"] = (
    gyeonggi_age["이용금액"]
    / gyeonggi_age["업종총이용금액"]
    * 100
).round(1)

gyeonggi_age.head(20)

In [ ]:
import numpy as np
import pandas as pd

# -----------------------------------------
# 1. 핵심 연령대 + 연령집중도
# -----------------------------------------

# 핵심 연령대
gyeonggi_dominant_age = gyeonggi_age.loc[
    gyeonggi_age.groupby(
        ["시군", "업종코드"]
    )["이용금액비중"].idxmax(),
    [
        "시군",
        "업종코드",
        "업종명",
        "연령대",
        "이용금액비중",
        "업종총이용금액"
    ]
].copy()

# 연령집중도(HHI)
gyeonggi_age["비중제곱"] = (
    gyeonggi_age["이용금액비중"] / 100
) ** 2

gyeonggi_concentration = (
    gyeonggi_age
    .groupby(
        ["시군", "업종코드", "업종명"],
        as_index=False
    )["비중제곱"]
    .sum()
    .rename(columns={"비중제곱": "연령집중도"})
)


# -----------------------------------------
# 2. 월별 이용금액
# -----------------------------------------

gyeonggi_monthly = (
    gyeonggi_bc
    .groupby(
        ["기준년월", "시군", "업종코드", "업종명"],
        as_index=False
    )["이용금액"]
    .sum()
)

gyeonggi_monthly_pivot = gyeonggi_monthly.pivot_table(
    index=["시군", "업종코드", "업종명"],
    columns="기준년월",
    values="이용금액"
).reset_index()

months = [
    202601, 202602, 202603,
    202604, 202605, 202606
]

# 보유 개월 수
gyeonggi_monthly_pivot["데이터보유개월수"] = (
    gyeonggi_monthly_pivot[months]
    .notna()
    .sum(axis=1)
)

gyeonggi_monthly_pivot["6개월완전"] = (
    gyeonggi_monthly_pivot["데이터보유개월수"] == 6
)

# 월별 증감률
monthly_pct = (
    gyeonggi_monthly_pivot[months]
    .pct_change(axis=1, fill_method=None)
    * 100
)

gyeonggi_monthly_pivot["월평균증감률"] = (
    monthly_pct.iloc[:, 1:]
    .mean(axis=1)
    .round(1)
)

gyeonggi_monthly_pivot["상승개월수"] = (
    monthly_pct.iloc[:, 1:] > 0
).sum(axis=1)

gyeonggi_monthly_pivot["1월대비6월증감률"] = (
    (
        gyeonggi_monthly_pivot[202606]
        - gyeonggi_monthly_pivot[202601]
    )
    / gyeonggi_monthly_pivot[202601]
    * 100
).round(1)

# 6개월 미완전 사례는 성장판정에서 제외
gyeonggi_monthly_pivot.loc[
    ~gyeonggi_monthly_pivot["6개월완전"],
    [
        "월평균증감률",
        "상승개월수",
        "1월대비6월증감률"
    ]
] = np.nan


# -----------------------------------------
# 3. 경기도 프로필 결합
# -----------------------------------------

gyeonggi_profile = (
    gyeonggi_dominant_age
    .merge(
        gyeonggi_concentration,
        on=["시군", "업종코드", "업종명"],
        how="left"
    )
    .merge(
        gyeonggi_monthly_pivot,
        on=["시군", "업종코드", "업종명"],
        how="left"
    )
)

gyeonggi_profile.head()

In [ ]:
# 경기도 미래인구 불러오기
gyeonggi_future = pd.read_csv(
    "../data/processed/gyeonggi_future_population_change.csv",
    encoding="utf-8-sig"
)

# BC 연령별 소비구조 + 미래인구 결합
gyeonggi_age_stress = gyeonggi_age.merge(
    gyeonggi_future[
        [
            "시군",
            "연령대",
            "2035 증감률",
            "2040 증감률"
        ]
    ],
    on=["시군", "연령대"],
    how="left"
)

print(
    "미래인구 결측 행 :",
    gyeonggi_age_stress["2035 증감률"].isna().sum()
)

In [ ]:
# 연령별 소비비중 × 미래 인구증감률
share = gyeonggi_age_stress["이용금액비중"] / 100

gyeonggi_age_stress["2035 순인구효과"] = (
    share * gyeonggi_age_stress["2035 증감률"]
)

gyeonggi_age_stress["2040 순인구효과"] = (
    share * gyeonggi_age_stress["2040 증감률"]
)

# 시군 × 업종별 전체 스트레스
gyeonggi_stress_summary = (
    gyeonggi_age_stress
    .groupby(
        ["시군", "업종코드", "업종명"],
        as_index=False
    )[
        ["2035 순인구효과", "2040 순인구효과"]
    ]
    .sum()
)

# 기존 프로필에 스트레스 붙이기
gyeonggi_result = gyeonggi_profile.merge(
    gyeonggi_stress_summary,
    on=["시군", "업종코드", "업종명"],
    how="left"
)

gyeonggi_result.head()

In [ ]:
# 경기도 내부 이용규모 상위 25% 기준
gyeonggi_q75 = gyeonggi_result["업종총이용금액"].quantile(0.75)

gyeonggi_same_base = gyeonggi_result[
    (gyeonggi_result["6개월완전"] == True) &
    (gyeonggi_result["업종총이용금액"] >= gyeonggi_q75)
].copy()

# 엄격한 성장조건
gyeonggi_same_base["엄격성장"] = (
    (gyeonggi_same_base["월평균증감률"] > 0) &
    (gyeonggi_same_base["1월대비6월증감률"] > 0) &
    (gyeonggi_same_base["상승개월수"] >= 3)
)

gyeonggi_growth = gyeonggi_same_base[
    gyeonggi_same_base["엄격성장"] == True
].copy()

gyeonggi_nongrowth = gyeonggi_same_base[
    gyeonggi_same_base["엄격성장"] == False
].copy()

print("경기도 상위25% 전체 :", len(gyeonggi_same_base))
print("경기도 현재성장 :", len(gyeonggi_growth))
print("경기도 비성장 :", len(gyeonggi_nongrowth))

In [ ]:
def city_group_summary(df, name):
    n = len(df)

    risk_2035 = (
        df["2035 순인구효과"] < 0
    ).sum()

    worse_2040 = (
        df["2040 순인구효과"]
        < df["2035 순인구효과"]
    ).sum()

    return {
        "구분": name,
        "사례수": n,
        "2035 위험사례": risk_2035,
        "2035 위험비율(%)":
            round(risk_2035 / n * 100, 1) if n > 0 else None,
        "2035 스트레스 중앙값":
            round(df["2035 순인구효과"].median(), 2) if n > 0 else None,
        "2040 추가악화": worse_2040,
        "2040 추가악화비율(%)":
            round(worse_2040 / n * 100, 1) if n > 0 else None,
        "2040 스트레스 중앙값":
            round(df["2040 순인구효과"].median(), 2) if n > 0 else None
    }


gyeonggi_compare = pd.DataFrame([
    city_group_summary(
        gyeonggi_same_base,
        "경기 상위25% 전체"
    ),
    city_group_summary(
        gyeonggi_growth,
        "경기 현재성장"
    ),
    city_group_summary(
        gyeonggi_nongrowth,
        "경기 비성장"
    )
])

gyeonggi_compare

In [ ]:
from scipy.stats import mannwhitneyu

# 2035
stat_2035_gg, p_2035_gg = mannwhitneyu(
    gyeonggi_growth["2035 순인구효과"],
    gyeonggi_nongrowth["2035 순인구효과"],
    alternative="less"
)

# 2040
stat_2040_gg, p_2040_gg = mannwhitneyu(
    gyeonggi_growth["2040 순인구효과"],
    gyeonggi_nongrowth["2040 순인구효과"],
    alternative="less"
)

print(
    "경기 2035 스트레스 차이 p-value :",
    round(p_2035_gg, 4)
)

print(
    "경기 2040 스트레스 차이 p-value :",
    round(p_2040_gg, 4)
)

In [ ]:
gyeonggi_compare

In [ ]:
import pandas as pd
from scipy.stats import fisher_exact, mannwhitneyu


def validate_city(city_result, city_name):

    # 1. 각 지역 내부 이용규모 상위 25%
    local_q75 = city_result["업종총이용금액"].quantile(0.75)

    base = city_result[
        (city_result["6개월완전"] == True) &
        (city_result["업종총이용금액"] >= local_q75)
    ].copy()

    # 2. 엄격한 성장조건
    base["엄격성장"] = (
        (base["월평균증감률"] > 0) &
        (base["1월대비6월증감률"] > 0) &
        (base["상승개월수"] >= 3)
    )

    growth = base[
        base["엄격성장"] == True
    ].copy()

    nongrowth = base[
        base["엄격성장"] == False
    ].copy()

    # 3. 위험 사례 수
    growth_risk = (
        growth["2035 순인구효과"] < 0
    ).sum()

    nongrowth_risk = (
        nongrowth["2035 순인구효과"] < 0
    ).sum()

    # 4. Fisher exact test
    fisher_table = [
        [growth_risk, len(growth) - growth_risk],
        [nongrowth_risk, len(nongrowth) - nongrowth_risk]
    ]

    _, fisher_p = fisher_exact(
        fisher_table,
        alternative="greater"
    )

    # 5. Mann-Whitney U
    _, p2035 = mannwhitneyu(
        growth["2035 순인구효과"],
        nongrowth["2035 순인구효과"],
        alternative="less"
    )

    _, p2040 = mannwhitneyu(
        growth["2040 순인구효과"],
        nongrowth["2040 순인구효과"],
        alternative="less"
    )

    # 6. 결과 요약
    summary = {
        "지역": city_name,

        "전체 사례": len(base),

        "성장군 n": len(growth),
        "비성장군 n": len(nongrowth),

        "성장군 2035 위험비율(%)":
            round(growth_risk / len(growth) * 100, 1),

        "비성장군 2035 위험비율(%)":
            round(nongrowth_risk / len(nongrowth) * 100, 1),

        "성장군 2035 중앙값":
            round(growth["2035 순인구효과"].median(), 2),

        "비성장군 2035 중앙값":
            round(nongrowth["2035 순인구효과"].median(), 2),

        "성장군 2040 중앙값":
            round(growth["2040 순인구효과"].median(), 2),

        "비성장군 2040 중앙값":
            round(nongrowth["2040 순인구효과"].median(), 2),

        "Fisher p":
            round(fisher_p, 4),

        "2035 MW p":
            round(p2035, 4),

        "2040 MW p":
            round(p2040, 4)
    }

    return summary, base, growth, nongrowth

In [ ]:
seoul_summary, seoul_base_local, seoul_growth_local, seoul_nongrowth_local = (
    validate_city(
        seoul_result,
        "서울"
    )
)

busan_summary, busan_base_local, busan_growth_local, busan_nongrowth_local = (
    validate_city(
        busan_result,
        "부산"
    )
)

gyeonggi_summary, gyeonggi_base_local, gyeonggi_growth_local, gyeonggi_nongrowth_local = (
    validate_city(
        gyeonggi_result,
        "경기"
    )
)

city_validation = pd.DataFrame([
    seoul_summary,
    busan_summary,
    gyeonggi_summary
])

city_validation

In [ ]:
import pandas as pd

# ------------------------------------------------
# 1. 서울 미래인구 데이터 다시 불러오기
# ------------------------------------------------

future_change = pd.read_csv(
    "../data/processed/seoul_future_population_change.csv",
    encoding="utf-8-sig"
)

future_seoul = future_change.copy()

# BC 연령대 이름과 맞추기
future_seoul["연령대"] = future_seoul["연령대"].replace({
    "20대미만": "20대이하"
})

future_seoul = future_seoul.rename(
    columns={"자치구별": "시군구"}
)


# ------------------------------------------------
# 2. 서울 BC 연령별 소비구조
# ------------------------------------------------

seoul_age_stress_restore = bc_age[
    bc_age["시도"] == "서울특별시"
].copy()

seoul_age_stress_restore = seoul_age_stress_restore.merge(
    future_seoul[
        [
            "시군구",
            "연령대",
            "2035 증감률",
            "2040 증감률"
        ]
    ],
    on=["시군구", "연령대"],
    how="left"
)

print(
    "서울 미래인구 결측 :",
    seoul_age_stress_restore["2035 증감률"].isna().sum()
)


# ------------------------------------------------
# 3. 미래 소비기반 스트레스 계산
# ------------------------------------------------

share = (
    seoul_age_stress_restore["이용금액비중"] / 100
)

seoul_age_stress_restore["2035 순인구효과"] = (
    share
    * seoul_age_stress_restore["2035 증감률"]
)

seoul_age_stress_restore["2040 순인구효과"] = (
    share
    * seoul_age_stress_restore["2040 증감률"]
)


# ------------------------------------------------
# 4. 지역×업종 단위로 합산
# ------------------------------------------------

seoul_stress_restore = (
    seoul_age_stress_restore
    .groupby(
        ["시도", "시군구", "업종코드", "업종명"],
        as_index=False
    )[
        ["2035 순인구효과", "2040 순인구효과"]
    ]
    .sum()
)


# ------------------------------------------------
# 5. 기존 BC 프로필과 결합
# ------------------------------------------------

seoul_result = (
    bc_profile[
        bc_profile["시도"] == "서울특별시"
    ]
    .merge(
        seoul_stress_restore,
        on=["시도", "시군구", "업종코드", "업종명"],
        how="left"
    )
)

print("서울 지역×업종 수 :", len(seoul_result))

seoul_result.head()

In [ ]:
seoul_summary, seoul_base_local, seoul_growth_local, seoul_nongrowth_local = (
    validate_city(
        seoul_result,
        "서울"
    )
)

busan_summary, busan_base_local, busan_growth_local, busan_nongrowth_local = (
    validate_city(
        busan_result,
        "부산"
    )
)

gyeonggi_summary, gyeonggi_base_local, gyeonggi_growth_local, gyeonggi_nongrowth_local = (
    validate_city(
        gyeonggi_result,
        "경기"
    )
)

city_validation = pd.DataFrame([
    seoul_summary,
    busan_summary,
    gyeonggi_summary
])

city_validation

In [ ]:
def classify_2x2(row):

    current_growth = (
        (row["월평균증감률"] > 0) &
        (row["1월대비6월증감률"] > 0) &
        (row["상승개월수"] >= 3)
    )

    future_stable = (
        row["2040 순인구효과"] >= 0
    )

    if current_growth and not future_stable:
        return "착시성장형"

    elif current_growth and future_stable:
        return "지속성장형"

    elif not current_growth and not future_stable:
        return "구조취약형"

    else:
        return "회복기회형"

In [ ]:
seoul_base_local["유형"] = (
    seoul_base_local.apply(classify_2x2, axis=1)
)

busan_base_local["유형"] = (
    busan_base_local.apply(classify_2x2, axis=1)
)

gyeonggi_base_local["유형"] = (
    gyeonggi_base_local.apply(classify_2x2, axis=1)
)

In [ ]:
type_summary = pd.DataFrame({
    "서울": seoul_base_local["유형"].value_counts(),
    "부산": busan_base_local["유형"].value_counts(),
    "경기": gyeonggi_base_local["유형"].value_counts()
}).fillna(0).astype(int)

type_summary

In [ ]:
all_type_cases = pd.concat([
    seoul_base_local.assign(지역="서울"),
    busan_base_local.assign(지역="부산"),
    gyeonggi_base_local.assign(지역="경기")
], ignore_index=True)

for type_name in [
    "착시성장형",
    "지속성장형",
    "구조취약형",
    "회복기회형"
]:
    print("\n====================")
    print(type_name)
    print("====================")

    display(
        all_type_cases[
            all_type_cases["유형"] == type_name
        ][
            [
                "지역",
                "시군구" if "시군구" in all_type_cases.columns else "시군",
                "업종명",
                "월평균증감률",
                "1월대비6월증감률",
                "2040 순인구효과"
            ]
        ].head(10)
    )

In [ ]:
# 지역명 컬럼 통일

seoul_cases = seoul_base_local.copy()
seoul_cases["광역지역"] = "서울"
seoul_cases["지역명"] = seoul_cases["시군구"]

busan_cases = busan_base_local.copy()
busan_cases["광역지역"] = "부산"
busan_cases["지역명"] = busan_cases["시군구"]

gyeonggi_cases = gyeonggi_base_local.copy()
gyeonggi_cases["광역지역"] = "경기"
gyeonggi_cases["지역명"] = gyeonggi_cases["시군"]

all_type_cases = pd.concat(
    [
        seoul_cases,
        busan_cases,
        gyeonggi_cases
    ],
    ignore_index=True
)

all_type_cases[
    [
        "광역지역",
        "지역명",
        "업종명",
        "유형",
        "월평균증감률",
        "1월대비6월증감률",
        "2040 순인구효과"
    ]
].head()

In [ ]:
# 착시성장형: 미래 위험이 큰 순
illusion_rep = (
    all_type_cases[
        all_type_cases["유형"] == "착시성장형"
    ]
    .sort_values(
        "2040 순인구효과",
        ascending=True
    )
    .head(3)
)

# 지속성장형: 미래 안정성이 높은 순
sustainable_rep = (
    all_type_cases[
        all_type_cases["유형"] == "지속성장형"
    ]
    .sort_values(
        "2040 순인구효과",
        ascending=False
    )
    .head(3)
)

# 구조취약형: 미래 위험이 큰 순
structural_rep = (
    all_type_cases[
        all_type_cases["유형"] == "구조취약형"
    ]
    .sort_values(
        "2040 순인구효과",
        ascending=True
    )
    .head(3)
)

# 회복기회형: 미래 안정성이 높은 순
recovery_rep = (
    all_type_cases[
        all_type_cases["유형"] == "회복기회형"
    ]
    .sort_values(
        "2040 순인구효과",
        ascending=False
    )
    .head(3)
)

In [ ]:
representative_4types = pd.concat([
    illusion_rep.assign(대표유형="착시성장형"),
    sustainable_rep.assign(대표유형="지속성장형"),
    structural_rep.assign(대표유형="구조취약형"),
    recovery_rep.assign(대표유형="회복기회형")
])

representative_4types[
    [
        "대표유형",
        "광역지역",
        "지역명",
        "업종명",
        "연령대",
        "월평균증감률",
        "1월대비6월증감률",
        "2035 순인구효과",
        "2040 순인구효과"
    ]
]

In [ ]:
pyeongtaek_case = gyeonggi_age_stress[
    (gyeonggi_age_stress["시군"] == "평택시") &
    (gyeonggi_age_stress["업종명"] == "일반한식")
].copy()

age_order = [
    "20대이하",
    "20대",
    "30대",
    "40대",
    "50대",
    "60대이상"
]

pyeongtaek_case["연령대"] = pd.Categorical(
    pyeongtaek_case["연령대"],
    categories=age_order,
    ordered=True
)

pyeongtaek_case = pyeongtaek_case.sort_values("연령대")

pyeongtaek_case[
    [
        "연령대",
        "이용금액비중",
        "2035 증감률",
        "2040 증감률",
        "2040 순인구효과"
    ]
]

In [ ]:
# 경기 일반한식 연령별 소비비중 75분위
gg_hansik_benchmark = (
    gyeonggi_age[
        gyeonggi_age["업종명"] == "일반한식"
    ]
    .groupby("연령대")["이용금액비중"]
    .quantile(0.75)
    .reset_index(name="동일업종_상위25%비중")
)

# 평택 사례에 benchmark 붙이기
pyeongtaek_expand = pyeongtaek_case.merge(
    gg_hansik_benchmark,
    on="연령대",
    how="left"
)

# 현재보다 어느 정도 더 확보할 여지가 있는가
pyeongtaek_expand["확장여력"] = (
    pyeongtaek_expand["동일업종_상위25%비중"]
    - pyeongtaek_expand["이용금액비중"]
).round(1)

# 현재 핵심고객
core_age = (
    pyeongtaek_expand
    .sort_values("이용금액비중", ascending=False)
    .iloc[0]["연령대"]
)

# 확장 후보
expand_candidates = pyeongtaek_expand[
    (pyeongtaek_expand["연령대"] != core_age) &
    (pyeongtaek_expand["확장여력"] > 0) &
    (pyeongtaek_expand["2040 증감률"] > 0)
].copy()

expand_candidates = expand_candidates.sort_values(
    ["확장여력", "2040 증감률"],
    ascending=[False, False]
)

print("현재 핵심고객 :", core_age)

expand_candidates[
    [
        "연령대",
        "이용금액비중",
        "동일업종_상위25%비중",
        "확장여력",
        "2040 증감률"
    ]
]

In [ ]:
hwaseong_case = gyeonggi_age_stress[
    (gyeonggi_age_stress["시군"] == "화성시") &
    (gyeonggi_age_stress["업종명"] == "슈퍼마켓")
].copy()

age_order = [
    "20대이하",
    "20대",
    "30대",
    "40대",
    "50대",
    "60대이상"
]

hwaseong_case["연령대"] = pd.Categorical(
    hwaseong_case["연령대"],
    categories=age_order,
    ordered=True
)

hwaseong_case = hwaseong_case.sort_values("연령대")

hwaseong_case[
    [
        "연령대",
        "이용금액비중",
        "2035 증감률",
        "2040 증감률",
        "2040 순인구효과"
    ]
]

In [ ]:
# 경기 슈퍼마켓 연령별 소비비중 75분위 benchmark
gg_super_benchmark = (
    gyeonggi_age[
        gyeonggi_age["업종명"] == "슈퍼마켓"
    ]
    .groupby("연령대")["이용금액비중"]
    .quantile(0.75)
    .reset_index(name="동일업종_상위25%비중")
)

# 화성시 슈퍼마켓에 benchmark 결합
hwaseong_preempt = hwaseong_case.merge(
    gg_super_benchmark,
    on="연령대",
    how="left"
)

# 현재보다 추가 확보 가능한 여력
hwaseong_preempt["확장여력"] = (
    hwaseong_preempt["동일업종_상위25%비중"]
    - hwaseong_preempt["이용금액비중"]
).round(1)

# 선점 잠재력
# = 현재 시장 확장여력 × 미래 인구증가 정도
hwaseong_preempt["선점잠재력"] = (
    hwaseong_preempt["확장여력"]
    * hwaseong_preempt["2040 증감률"].clip(lower=0)
    / 100
).round(2)

# 현재 핵심고객
hwaseong_core_age = (
    hwaseong_preempt
    .sort_values("이용금액비중", ascending=False)
    .iloc[0]["연령대"]
)

# 선점 후보
preempt_candidates = hwaseong_preempt[
    (hwaseong_preempt["연령대"] != hwaseong_core_age) &
    (hwaseong_preempt["확장여력"] > 0) &
    (hwaseong_preempt["2040 증감률"] > 0)
].copy()

preempt_candidates = preempt_candidates.sort_values(
    "선점잠재력",
    ascending=False
)

print("현재 핵심고객 :", hwaseong_core_age)

preempt_candidates[
    [
        "연령대",
        "이용금액비중",
        "동일업종_상위25%비중",
        "확장여력",
        "2040 증감률",
        "선점잠재력"
    ]
]

In [ ]:
# 경기도 회복기회형 사례
recovery_cases = gyeonggi_base_local[
    gyeonggi_base_local["유형"] == "회복기회형"
][
    ["시군", "업종코드", "업종명"]
].copy()

# 연령별 소비구조 + 미래인구 연결
recovery_pool = gyeonggi_age_stress.merge(
    recovery_cases,
    on=["시군", "업종코드", "업종명"],
    how="inner"
)

# 동일 업종 × 연령대 소비비중 75분위 benchmark
gg_benchmark = (
    gyeonggi_age
    .groupby(
        ["업종코드", "업종명", "연령대"],
        as_index=False
    )["이용금액비중"]
    .quantile(0.75)
    .rename(
        columns={"이용금액비중": "동일업종_상위25%비중"}
    )
)

recovery_pool = recovery_pool.merge(
    gg_benchmark,
    on=["업종코드", "업종명", "연령대"],
    how="left"
)

# 현재 핵심고객 찾기
core_age_table = (
    recovery_pool.loc[
        recovery_pool.groupby(
            ["시군", "업종코드"]
        )["이용금액비중"].idxmax(),
        ["시군", "업종코드", "연령대"]
    ]
    .rename(columns={"연령대": "현재핵심고객"})
)

recovery_pool = recovery_pool.merge(
    core_age_table,
    on=["시군", "업종코드"],
    how="left"
)

# 확장여력
recovery_pool["확장여력"] = (
    recovery_pool["동일업종_상위25%비중"]
    - recovery_pool["이용금액비중"]
).round(1)

# 선점잠재력
recovery_pool["선점잠재력"] = (
    recovery_pool["확장여력"]
    * recovery_pool["2040 증감률"].clip(lower=0)
    / 100
).round(2)

# 선점 후보만
recovery_candidates = recovery_pool[
    (recovery_pool["연령대"] != recovery_pool["현재핵심고객"]) &
    (recovery_pool["확장여력"] > 0) &
    (recovery_pool["2040 증감률"] > 0)
].copy()

# 각 지역×업종에서 가장 좋은 선점고객 1명
best_recovery = (
    recovery_candidates
    .sort_values("선점잠재력", ascending=False)
    .drop_duplicates(
        subset=["시군", "업종코드"]
    )
)

best_recovery[
    [
        "시군",
        "업종명",
        "현재핵심고객",
        "연령대",
        "이용금액비중",
        "동일업종_상위25%비중",
        "확장여력",
        "2040 증감률",
        "선점잠재력"
    ]
].head(10)

In [ ]:
# 부산 사상구 편의점 연령별 구조
sasang_case = busan_age_stress[
    (busan_age_stress["시군구"] == "사상구") &
    (busan_age_stress["업종명"] == "편의점")
].copy()

age_order = [
    "20대이하",
    "20대",
    "30대",
    "40대",
    "50대",
    "60대이상"
]

sasang_case["연령대"] = pd.Categorical(
    sasang_case["연령대"],
    categories=age_order,
    ordered=True
)

sasang_case = sasang_case.sort_values("연령대")

sasang_case[
    [
        "연령대",
        "이용금액비중",
        "2035 증감률",
        "2040 증감률",
        "2040 순인구효과"
    ]
]

In [ ]:
# 부산 편의점의 연령대별 소비비중 75분위 benchmark
busan_store_benchmark = (
    busan_age_stress[
        busan_age_stress["업종명"] == "편의점"
    ]
    .groupby("연령대")["이용금액비중"]
    .quantile(0.75)
    .reset_index(name="동일업종_상위25%비중")
)

# 사상구 편의점에 benchmark 결합
sasang_defense = sasang_case.merge(
    busan_store_benchmark,
    on="연령대",
    how="left"
)

# 현재 핵심고객
sasang_core_age = (
    sasang_defense
    .sort_values("이용금액비중", ascending=False)
    .iloc[0]["연령대"]
)

# 핵심고객의 2040 인구증감률
core_2040_change = (
    sasang_defense[
        sasang_defense["연령대"] == sasang_core_age
    ]["2040 증감률"]
    .iloc[0]
)

# 추가 확보 가능한 소비비중
sasang_defense["확장여력"] = (
    sasang_defense["동일업종_상위25%비중"]
    - sasang_defense["이용금액비중"]
).round(1)

# 현재 핵심고객보다 미래 인구구조가 얼마나 유리한가
sasang_defense["2040 인구상대우위"] = (
    sasang_defense["2040 증감률"]
    - core_2040_change
).round(1)

# 방어 잠재력
sasang_defense["방어잠재력"] = (
    sasang_defense["확장여력"]
    * sasang_defense["2040 인구상대우위"].clip(lower=0)
    / 100
).round(2)

# 방어 후보
sasang_candidates = sasang_defense[
    (sasang_defense["연령대"] != sasang_core_age) &
    (sasang_defense["확장여력"] > 0) &
    (sasang_defense["2040 인구상대우위"] > 0)
].copy()

sasang_candidates = sasang_candidates.sort_values(
    "방어잠재력",
    ascending=False
)

print("현재 핵심고객 :", sasang_core_age)

sasang_candidates[
    [
        "연령대",
        "이용금액비중",
        "동일업종_상위25%비중",
        "확장여력",
        "2040 증감률",
        "2040 인구상대우위",
        "방어잠재력"
    ]
]

In [ ]:
sasang_current_2040 = (
    busan_base_local[
        (busan_base_local["시군구"] == "사상구") &
        (busan_base_local["업종명"] == "편의점")
    ]["2040 순인구효과"]
    .iloc[0]
)

best_defense = sasang_candidates.iloc[0]

sasang_after_2040 = (
    sasang_current_2040
    + best_defense["방어잠재력"]
)

sasang_mitigation = (
    best_defense["방어잠재력"]
    / abs(sasang_current_2040)
    * 100
)

print("2040 전환 전 :", round(sasang_current_2040, 2))
print("2040 전환 후 :", round(sasang_after_2040, 2))
print("위험완화율 :", round(sasang_mitigation, 1), "%")

In [ ]:
all_type_cases.to_csv(
    "../data/processed/seoul_busan_gyeonggi_2x2_types.csv",
    index=False,
    encoding="utf-8-sig"
)

representative_4types.to_csv(
    "../data/processed/representative_4types.csv",
    index=False,
    encoding="utf-8-sig"
)

print("4유형 분석결과 저장 완료")

In [ ]:
import pandas as pd

# ------------------------------------------------
# 1. 필요한 저장 데이터 다시 불러오기
# ------------------------------------------------

bc_age = pd.read_csv(
    "../data/processed/bc_age_consumption_structure.csv",
    encoding="utf-8-sig"
)

bc_profile = pd.read_csv(
    "../data/processed/bc_region_industry_profile.csv",
    encoding="utf-8-sig"
)

busan_future = pd.read_csv(
    "../data/processed/busan_future_population_change.csv",
    encoding="utf-8-sig"
)


# ------------------------------------------------
# 2. 부산 미래 소비기반 스트레스 다시 계산
# ------------------------------------------------

busan_age_restore = bc_age[
    bc_age["시도"] == "부산광역시"
].copy()

busan_age_restore = busan_age_restore.merge(
    busan_future[
        [
            "시군구",
            "연령대",
            "2035 증감률",
            "2040 증감률"
        ]
    ],
    on=["시군구", "연령대"],
    how="left"
)

share = busan_age_restore["이용금액비중"] / 100

busan_age_restore["2035 순인구효과"] = (
    share * busan_age_restore["2035 증감률"]
)

busan_age_restore["2040 순인구효과"] = (
    share * busan_age_restore["2040 증감률"]
)

busan_stress_restore = (
    busan_age_restore
    .groupby(
        ["시도", "시군구", "업종코드", "업종명"],
        as_index=False
    )[
        ["2035 순인구효과", "2040 순인구효과"]
    ]
    .sum()
)


# ------------------------------------------------
# 3. 부산 프로필과 결합
# ------------------------------------------------

busan_result = (
    bc_profile[
        bc_profile["시도"] == "부산광역시"
    ]
    .merge(
        busan_stress_restore,
        on=["시도", "시군구", "업종코드", "업종명"],
        how="left"
    )
)


# ------------------------------------------------
# 4. 부산 내부 상위 25% 기준 복구
# ------------------------------------------------

busan_q75 = (
    busan_result["업종총이용금액"]
    .quantile(0.75)
)

busan_base_local = busan_result[
    (busan_result["6개월완전"] == True) &
    (busan_result["업종총이용금액"] >= busan_q75)
].copy()

print("부산 비교대상 사례 수 :", len(busan_base_local))

In [ ]:
busan_living_compare = busan_base_local.merge(
    busan_living_summary,
    on=["시도", "시군구", "업종코드", "업종명"],
    how="left"
)

print(
    "생활인구 보정 결측 :",
    busan_living_compare["2040 생활인구보정효과"]
    .isna()
    .sum()
)

In [ ]:
import pandas as pd

# =========================================================
# 1. 부산 생활인구 전체 파일 읽기
# =========================================================

busan_living_raw = pd.read_excel(
    "../data/raw/busan_living_population_age.xlsx"
)

# 2025년 자료 사용
busan_living = busan_living_raw[
    busan_living_raw["기준년월"]
    .astype(str)
    .str.startswith("2025")
].copy()

# 행정동명 예: "중구 중앙동" -> "중구"
busan_living["시군구"] = (
    busan_living["행정동명"]
    .astype(str)
    .str.split()
    .str[0]
)

# BC 연령대와 통일
busan_living["연령대"] = (
    busan_living["나이대"]
    .replace({
        "20대미만": "20대이하"
    })
)


# =========================================================
# 2. 구군 × 연령대별 생활인구 계산
# =========================================================

# 먼저 월별로 행정동 합산
busan_living_age = (
    busan_living
    .groupby(
        ["기준년월", "시군구", "연령대"],
        as_index=False
    )[
        [
            "평균주거인구수",
            "평균직장인구수",
            "평균방문인구수"
        ]
    ]
    .sum()
)

# 2025년 월평균
busan_living_age = (
    busan_living_age
    .groupby(
        ["시군구", "연령대"],
        as_index=False
    )[
        [
            "평균주거인구수",
            "평균직장인구수",
            "평균방문인구수"
        ]
    ]
    .mean()
)

# 총 생활인구
busan_living_age["총생활인구"] = (
    busan_living_age["평균주거인구수"]
    + busan_living_age["평균직장인구수"]
    + busan_living_age["평균방문인구수"]
)

# 거주인구 노출비율
busan_living_age["거주노출비율"] = (
    busan_living_age["평균주거인구수"]
    / busan_living_age["총생활인구"]
)


# =========================================================
# 3. 부산 BC 소비구조 + 미래인구 다시 결합
# =========================================================

bc_age = pd.read_csv(
    "../data/processed/bc_age_consumption_structure.csv",
    encoding="utf-8-sig"
)

busan_future = pd.read_csv(
    "../data/processed/busan_future_population_change.csv",
    encoding="utf-8-sig"
)

busan_age_stress = bc_age[
    bc_age["시도"] == "부산광역시"
].copy()

busan_age_stress = busan_age_stress.merge(
    busan_future[
        [
            "시군구",
            "연령대",
            "2035 증감률",
            "2040 증감률"
        ]
    ],
    on=["시군구", "연령대"],
    how="left"
)


# =========================================================
# 4. 생활인구 보정계수 붙이기
# =========================================================

busan_living_stress = busan_age_stress.merge(
    busan_living_age[
        [
            "시군구",
            "연령대",
            "거주노출비율"
        ]
    ],
    on=["시군구", "연령대"],
    how="left"
)

print(
    "생활인구 결측 :",
    busan_living_stress["거주노출비율"].isna().sum()
)


# =========================================================
# 5. 생활인구 보정 스트레스 계산
# =========================================================

share = busan_living_stress["이용금액비중"] / 100

busan_living_stress["2035 생활인구보정효과"] = (
    share
    * busan_living_stress["2035 증감률"]
    * busan_living_stress["거주노출비율"]
)

busan_living_stress["2040 생활인구보정효과"] = (
    share
    * busan_living_stress["2040 증감률"]
    * busan_living_stress["거주노출비율"]
)

busan_living_summary = (
    busan_living_stress
    .groupby(
        ["시도", "시군구", "업종코드", "업종명"],
        as_index=False
    )[
        [
            "2035 생활인구보정효과",
            "2040 생활인구보정효과"
        ]
    ]
    .sum()
)

print("생활인구 보정 결과 수 :", len(busan_living_summary))

busan_living_summary.head()

In [ ]:
busan_living_compare = busan_base_local.merge(
    busan_living_summary,
    on=["시도", "시군구", "업종코드", "업종명"],
    how="left"
)

print(
    "생활인구 보정 결측 :",
    busan_living_compare["2040 생활인구보정효과"]
    .isna()
    .sum()
)

In [ ]:
from scipy.stats import spearmanr

# 기존 / 생활인구 보정 후 위험 여부
busan_living_compare["기존_2040위험"] = (
    busan_living_compare["2040 순인구효과"] < 0
)

busan_living_compare["보정_2040위험"] = (
    busan_living_compare["2040 생활인구보정효과"] < 0
)

# 위험 판정 일치
busan_living_compare["위험판정일치"] = (
    busan_living_compare["기존_2040위험"]
    == busan_living_compare["보정_2040위험"]
)

agreement_rate = (
    busan_living_compare["위험판정일치"].mean() * 100
)

changed_cases = (
    ~busan_living_compare["위험판정일치"]
).sum()

print(
    "2040 위험/안정 판정 일치율 :",
    round(agreement_rate, 1),
    "%"
)

print(
    "판정 변경 사례 :",
    changed_cases,
    "/",
    len(busan_living_compare)
)

# 위험 순위 상관
corr, p_value = spearmanr(
    busan_living_compare["2040 순인구효과"],
    busan_living_compare["2040 생활인구보정효과"]
)

print(
    "2040 스트레스 순위 상관 :",
    round(corr, 3)
)

print(
    "Spearman p-value :",
    round(p_value, 4)
)

In [ ]:
# 엄격한 현재 성장조건
busan_living_compare["현재성장"] = (
    (busan_living_compare["월평균증감률"] > 0) &
    (busan_living_compare["1월대비6월증감률"] > 0) &
    (busan_living_compare["상승개월수"] >= 3)
)

def make_type(growth, future_effect):
    if growth and future_effect < 0:
        return "착시성장형"
    elif growth and future_effect >= 0:
        return "지속성장형"
    elif not growth and future_effect < 0:
        return "구조취약형"
    else:
        return "회복기회형"

# 기존 유형
busan_living_compare["기존유형"] = busan_living_compare.apply(
    lambda row: make_type(
        row["현재성장"],
        row["2040 순인구효과"]
    ),
    axis=1
)

# 생활인구 보정 유형
busan_living_compare["보정유형"] = busan_living_compare.apply(
    lambda row: make_type(
        row["현재성장"],
        row["2040 생활인구보정효과"]
    ),
    axis=1
)

busan_living_compare["유형일치"] = (
    busan_living_compare["기존유형"]
    == busan_living_compare["보정유형"]
)

print(
    "4유형 판정 일치율 :",
    round(
        busan_living_compare["유형일치"].mean() * 100,
        1
    ),
    "%"
)

print(
    "유형 변경 사례 :",
    (~busan_living_compare["유형일치"]).sum(),
    "/",
    len(busan_living_compare)
)

In [ ]:
changed_cases = busan_living_compare[
    busan_living_compare["유형일치"] == False
].copy()

changed_cases[
    [
        "시군구",
        "업종명",
        "월평균증감률",
        "1월대비6월증감률",
        "2040 순인구효과",
        "2040 생활인구보정효과",
        "기존유형",
        "보정유형"
    ]
]

In [ ]:
import pandas as pd

# -----------------------------------------
# 1. 데이터 다시 불러오기
# -----------------------------------------

bc_age = pd.read_csv(
    "../data/processed/bc_age_consumption_structure.csv",
    encoding="utf-8-sig"
)

future_change = pd.read_csv(
    "../data/processed/seoul_future_population_change.csv",
    encoding="utf-8-sig"
)

future_seoul = future_change.copy()

future_seoul["연령대"] = future_seoul["연령대"].replace({
    "20대미만": "20대이하"
})

future_seoul = future_seoul.rename(
    columns={"자치구별": "시군구"}
)


# -----------------------------------------
# 2. 서울 서양음식의 연령별 소비비중 벤치마크
#    25% ~ 75% 범위
# -----------------------------------------

western_benchmark = (
    bc_age[
        (bc_age["시도"] == "서울특별시") &
        (bc_age["업종명"] == "서양음식")
    ]
    .groupby("연령대")["이용금액비중"]
    .quantile([0.25, 0.75])
    .unstack()
    .reset_index()
    .rename(columns={
        0.25: "동일업종_Q25",
        0.75: "동일업종_Q75"
    })
)


# -----------------------------------------
# 3. 성동구 서양음식 현재 고객구조
# -----------------------------------------

rebalance_case = bc_age[
    (bc_age["시도"] == "서울특별시") &
    (bc_age["시군구"] == "성동구") &
    (bc_age["업종명"] == "서양음식")
][
    ["연령대", "이용금액비중"]
].copy()

rebalance_case = rebalance_case.merge(
    future_seoul[
        ["시군구", "연령대", "2040 증감률"]
    ],
    left_on="연령대",
    right_on="연령대",
    how="left"
).drop(columns="시군구")

rebalance_case = rebalance_case.merge(
    western_benchmark,
    on="연령대",
    how="left"
)


# -----------------------------------------
# 4. 현실적인 조정 가능 범위
# -----------------------------------------

# 현재보다 Q25 아래로 억지로 내리지 않음
rebalance_case["최소비중"] = rebalance_case[
    ["이용금액비중", "동일업종_Q25"]
].min(axis=1)

# 현재보다 Q75 위로 억지로 올리지 않음
rebalance_case["최대비중"] = rebalance_case[
    ["이용금액비중", "동일업종_Q75"]
].max(axis=1)

# 현재 연령별 2040 기여도
rebalance_case["현재_2040기여"] = (
    rebalance_case["이용금액비중"] / 100
    * rebalance_case["2040 증감률"]
).round(2)

age_order = [
    "20대이하", "20대", "30대",
    "40대", "50대", "60대이상"
]

rebalance_case["연령대"] = pd.Categorical(
    rebalance_case["연령대"],
    categories=age_order,
    ordered=True
)

rebalance_case = rebalance_case.sort_values("연령대")

rebalance_case[
    [
        "연령대",
        "이용금액비중",
        "2040 증감률",
        "동일업종_Q25",
        "동일업종_Q75",
        "최소비중",
        "최대비중",
        "현재_2040기여"
    ]
]

In [ ]:
# -----------------------------------------
# 성동구 서양음식 고객 포트폴리오 다시 생성
# -----------------------------------------

rebalance_case = bc_age[
    (bc_age["시도"] == "서울특별시") &
    (bc_age["시군구"] == "성동구") &
    (bc_age["업종명"] == "서양음식")
][
    ["연령대", "이용금액비중"]
].copy()


# 성동구 미래인구만 먼저 필터링
seongdong_future = future_seoul[
    future_seoul["시군구"] == "성동구"
][
    ["연령대", "2040 증감률"]
].copy()


# 연령대 기준으로 결합
rebalance_case = rebalance_case.merge(
    seongdong_future,
    on="연령대",
    how="left"
)


# 동일업종 benchmark 결합
rebalance_case = rebalance_case.merge(
    western_benchmark,
    on="연령대",
    how="left"
)


# 현실적인 조정 가능 범위
rebalance_case["최소비중"] = rebalance_case[
    ["이용금액비중", "동일업종_Q25"]
].min(axis=1)

rebalance_case["최대비중"] = rebalance_case[
    ["이용금액비중", "동일업종_Q75"]
].max(axis=1)


# 현재 2040 기여도
rebalance_case["현재_2040기여"] = (
    rebalance_case["이용금액비중"] / 100
    * rebalance_case["2040 증감률"]
).round(2)


age_order = [
    "20대이하",
    "20대",
    "30대",
    "40대",
    "50대",
    "60대이상"
]

rebalance_case["연령대"] = pd.Categorical(
    rebalance_case["연령대"],
    categories=age_order,
    ordered=True
)

rebalance_case = (
    rebalance_case
    .sort_values("연령대")
    .reset_index(drop=True)
)


rebalance_case[
    [
        "연령대",
        "이용금액비중",
        "2040 증감률",
        "동일업종_Q25",
        "동일업종_Q75",
        "최소비중",
        "최대비중",
        "현재_2040기여"
    ]
]

In [ ]:
print("행 수 :", len(rebalance_case))
print(
    "현재 고객비중 합 :",
    round(rebalance_case["이용금액비중"].sum(), 1),
    "%"
)

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# -----------------------------------------
# 1. 현재 비중을 정확히 100%로 정규화
# -----------------------------------------

portfolio = rebalance_case.copy()

portfolio["현재비중"] = (
    portfolio["이용금액비중"]
    / portfolio["이용금액비중"].sum()
    * 100
)

current = portfolio["현재비중"].to_numpy()

future_change = (
    portfolio["2040 증감률"].to_numpy()
)

lower = portfolio["최소비중"].to_numpy()
upper = portfolio["최대비중"].to_numpy()

n = len(portfolio)


# -----------------------------------------
# 2. 주어진 '총 이동량' 안에서
#    2040 스트레스를 가장 많이 개선하는 함수
# -----------------------------------------

def optimize_rebalance(max_shift_pp):

    # 변수:
    # x[0:n] = 새로운 연령별 고객비중
    # d[0:n] = 현재 비중과의 절대 차이

    # 목적함수:
    # 2040 순인구효과 최대화
    # linprog는 최소화 문제라 음수 사용
    c = np.concatenate([
        -future_change / 100,
        np.zeros(n)
    ])

    A_ub = []
    b_ub = []

    # |x - current| <= d
    for i in range(n):

        row1 = np.zeros(2*n)
        row1[i] = 1
        row1[n+i] = -1
        A_ub.append(row1)
        b_ub.append(current[i])

        row2 = np.zeros(2*n)
        row2[i] = -1
        row2[n+i] = -1
        A_ub.append(row2)
        b_ub.append(-current[i])

    # 실제 이동량 = 절대변화합 / 2
    # 따라서 d 합 <= 2 × 허용 이동량
    shift_row = np.zeros(2*n)
    shift_row[n:] = 1

    A_ub.append(shift_row)
    b_ub.append(2 * max_shift_pp)

    # 새로운 고객비중 합 = 100
    A_eq = np.zeros((1, 2*n))
    A_eq[0, :n] = 1

    b_eq = [100]

    bounds = (
        list(zip(lower, upper))
        + [(0, None)] * n
    )

    result = linprog(
        c,
        A_ub=np.array(A_ub),
        b_ub=np.array(b_ub),
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs"
    )

    if not result.success:
        return None

    new_share = result.x[:n]

    before_stress = (
        current * future_change / 100
    ).sum()

    after_stress = (
        new_share * future_change / 100
    ).sum()

    actual_shift = (
        np.abs(new_share - current).sum() / 2
    )

    mitigation = (
        (after_stress - before_stress)
        / abs(before_stress)
        * 100
    )

    return {
        "허용이동량": max_shift_pp,
        "실제이동량": round(actual_shift, 2),
        "전환전_2040": round(before_stress, 2),
        "전환후_2040": round(after_stress, 2),
        "위험완화율": round(mitigation, 1),
        "새비중": new_share
    }

In [ ]:
results = []

for shift in range(1, 11):
    r = optimize_rebalance(shift)

    if r is not None:
        results.append({
            "고객비중 이동량(%p)": r["실제이동량"],
            "전환 전 2040 스트레스": r["전환전_2040"],
            "전환 후 2040 스트레스": r["전환후_2040"],
            "위험완화율(%)": r["위험완화율"]
        })

rebalance_frontier = pd.DataFrame(results)

rebalance_frontier

In [ ]:
scenario_5 = optimize_rebalance(5)

portfolio_result = portfolio[
    ["연령대", "현재비중"]
].copy()

portfolio_result["추천비중"] = (
    scenario_5["새비중"]
).round(1)

portfolio_result["변화(%p)"] = (
    portfolio_result["추천비중"]
    - portfolio_result["현재비중"]
).round(1)

print(
    "2040 스트레스:",
    scenario_5["전환전_2040"],
    "→",
    scenario_5["전환후_2040"]
)

print(
    "위험완화율:",
    scenario_5["위험완화율"],
    "%"
)

portfolio_result

In [ ]:
check_vars = [
    "all_type_cases",
    "seoul_age_stress_restore",
    "busan_age_stress",
    "gyeonggi_age_stress"
]

for v in check_vars:
    print(v, ":", v in globals())

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# =====================================================
# 1. 서울·부산·경기 연령별 분석데이터 통일
# =====================================================

seoul_pool = seoul_age_stress_restore.copy()
seoul_pool["광역지역"] = "서울"
seoul_pool["지역명"] = seoul_pool["시군구"]

busan_pool = busan_age_stress.copy()
busan_pool["광역지역"] = "부산"
busan_pool["지역명"] = busan_pool["시군구"]

gyeonggi_pool = gyeonggi_age_stress.copy()
gyeonggi_pool["광역지역"] = "경기"
gyeonggi_pool["지역명"] = gyeonggi_pool["시군"]

age_pool = pd.concat(
    [seoul_pool, busan_pool, gyeonggi_pool],
    ignore_index=True
)

# =====================================================
# 2. 지역별 동일업종 연령비중 Q25 / Q75
# =====================================================

benchmark = (
    age_pool
    .groupby(
        ["광역지역", "업종코드", "업종명", "연령대"]
    )["이용금액비중"]
    .quantile([0.25, 0.75])
    .unstack()
    .reset_index()
    .rename(columns={
        0.25: "동일업종_Q25",
        0.75: "동일업종_Q75"
    })
)

# =====================================================
# 3. 착시성장형 17건만 추출
# =====================================================

illusion_cases = all_type_cases[
    all_type_cases["유형"] == "착시성장형"
][
    [
        "광역지역",
        "지역명",
        "업종코드",
        "업종명"
    ]
].drop_duplicates()

print("착시성장 사례 수 :", len(illusion_cases))

# =====================================================
# 4. 한 사례의 고객포트폴리오 최적화 함수
# =====================================================

def optimize_case(case_row, max_shift_pp=5):

    region = case_row["광역지역"]
    area = case_row["지역명"]
    code = case_row["업종코드"]
    industry = case_row["업종명"]

    p = age_pool[
        (age_pool["광역지역"] == region) &
        (age_pool["지역명"] == area) &
        (age_pool["업종코드"] == code)
    ][
        [
            "연령대",
            "이용금액비중",
            "2040 증감률"
        ]
    ].copy()

    p = p.merge(
        benchmark[
            (benchmark["광역지역"] == region) &
            (benchmark["업종코드"] == code)
        ][
            [
                "연령대",
                "동일업종_Q25",
                "동일업종_Q75"
            ]
        ],
        on="연령대",
        how="left"
    )

    if len(p) == 0:
        return None

    # 현재 비중을 정확히 100으로 정규화
    p["현재비중"] = (
        p["이용금액비중"]
        / p["이용금액비중"].sum()
        * 100
    )

    # 실제 동일업종에서 관측된 범위 안에서만 조정
    p["최소비중"] = p[
        ["현재비중", "동일업종_Q25"]
    ].min(axis=1)

    p["최대비중"] = p[
        ["현재비중", "동일업종_Q75"]
    ].max(axis=1)

    current = p["현재비중"].to_numpy()
    future = p["2040 증감률"].to_numpy()

    lower = p["최소비중"].to_numpy()
    upper = p["최대비중"].to_numpy()

    n = len(p)

    # x = 추천 고객비중
    # d = 현재와 추천의 절대 차이
    c = np.concatenate([
        -future / 100,
        np.zeros(n)
    ])

    A_ub = []
    b_ub = []

    for i in range(n):

        row1 = np.zeros(2*n)
        row1[i] = 1
        row1[n+i] = -1
        A_ub.append(row1)
        b_ub.append(current[i])

        row2 = np.zeros(2*n)
        row2[i] = -1
        row2[n+i] = -1
        A_ub.append(row2)
        b_ub.append(-current[i])

    # 고객비중 총 이동량 <= 5%p
    shift_row = np.zeros(2*n)
    shift_row[n:] = 1

    A_ub.append(shift_row)
    b_ub.append(2 * max_shift_pp)

    # 추천 비중 합 = 100
    A_eq = np.zeros((1, 2*n))
    A_eq[0, :n] = 1

    result = linprog(
        c,
        A_ub=np.array(A_ub),
        b_ub=np.array(b_ub),
        A_eq=A_eq,
        b_eq=[100],
        bounds=(
            list(zip(lower, upper))
            + [(0, None)] * n
        ),
        method="highs"
    )

    if not result.success:
        return None

    new_share = result.x[:n]
    change = new_share - current

    before = (
        current * future / 100
    ).sum()

    after = (
        new_share * future / 100
    ).sum()

    mitigation = (
        (after - before)
        / abs(before)
        * 100
    )

    increased = p.loc[
        change > 0.05,
        "연령대"
    ].astype(str).tolist()

    return {
        "광역지역": region,
        "지역명": area,
        "업종명": industry,

        "최대감소연령":
            str(p.iloc[np.argmin(change)]["연령대"]),

        "최대증가연령":
            str(p.iloc[np.argmax(change)]["연령대"]),

        "증가연령전체":
            ", ".join(increased),

        "실제이동량(%p)":
            round(np.abs(change).sum() / 2, 2),

        "2040_전환전":
            round(before, 2),

        "2040_전환후":
            round(after, 2),

        "위험완화율(%)":
            round(mitigation, 1)
    }

# =====================================================
# 5. 착시성장 전체에 적용
# =====================================================

results = []

for _, row in illusion_cases.iterrows():

    r = optimize_case(
        row,
        max_shift_pp=5
    )

    if r is not None:
        results.append(r)

illusion_rebalance = pd.DataFrame(results)

illusion_rebalance

In [ ]:
age_result = (
    illusion_rebalance["최대증가연령"]
    .value_counts()
    .reset_index()
)

age_result.columns = [
    "최적 증가 연령대",
    "사례수"
]

age_result["비율(%)"] = (
    age_result["사례수"]
    / len(illusion_rebalance)
    * 100
).round(1)

age_result

In [ ]:
print(
    "60대 이상이 최적 증가연령인 사례 :",
    (
        illusion_rebalance["최대증가연령"]
        == "60대이상"
    ).sum(),
    "/",
    len(illusion_rebalance)
)

print(
    "평균 위험완화율 :",
    round(
        illusion_rebalance["위험완화율(%)"].mean(),
        1
    ),
    "%"
)

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# 기존 rebalance_case 사용
portfolio = rebalance_case.copy()

# 현재 비중 정확히 100%로 정규화
portfolio["현재비중"] = (
    portfolio["이용금액비중"]
    / portfolio["이용금액비중"].sum()
    * 100
)

current = portfolio["현재비중"].to_numpy()
future = portfolio["2040 증감률"].to_numpy()

# 현재값과 동일업종 Q25/Q75를 이용한 현실적 범위
q25 = portfolio["동일업종_Q25"].to_numpy()
q75 = portfolio["동일업종_Q75"].to_numpy()

lower = np.minimum(current, q25)
upper = np.maximum(current, q75)

# 현재 스트레스
current_stress = np.sum(
    current * future / 100
)

# 현재 연령집중도(HHI)
current_hhi = np.sum(
    (current / 100) ** 2
)


def diversified_rebalance(target_mitigation_pct):

    # 목표 스트레스
    # 예: 10% 완화라면 -17.24의 10%만큼 0 방향으로 이동
    target_stress = (
        current_stress
        + abs(current_stress)
        * target_mitigation_pct / 100
    )

    # 목적:
    # 현재 고객구조에서 변화량을 가능한 작게
    # 제곱합을 쓰면 한 연령에 몰아주는 것보다 분산이 유리함
    def objective(x):
        return np.sum((x - current) ** 2)

    constraints = [

        # 비중 합 = 100%
        {
            "type": "eq",
            "fun": lambda x: np.sum(x) - 100
        },

        # 목표 위험완화 이상 달성
        {
            "type": "ineq",
            "fun": lambda x:
                np.sum(x * future / 100)
                - target_stress
        },

        # HHI가 현재보다 악화되지 않음
        {
            "type": "ineq",
            "fun": lambda x:
                current_hhi
                - np.sum((x / 100) ** 2)
        }
    ]

    result = minimize(
        objective,
        x0=current,
        method="SLSQP",
        bounds=list(zip(lower, upper)),
        constraints=constraints,
        options={
            "maxiter": 2000,
            "ftol": 1e-9
        }
    )

    if not result.success:
        return None

    new_share = result.x

    after_stress = np.sum(
        new_share * future / 100
    )

    actual_mitigation = (
        (after_stress - current_stress)
        / abs(current_stress)
        * 100
    )

    new_hhi = np.sum(
        (new_share / 100) ** 2
    )

    total_shift = (
        np.abs(new_share - current).sum()
        / 2
    )

    return {
        "목표완화율": target_mitigation_pct,
        "실제완화율": round(actual_mitigation, 1),
        "이동량": round(total_shift, 2),
        "전환전": round(current_stress, 2),
        "전환후": round(after_stress, 2),
        "기존HHI": round(current_hhi, 4),
        "추천HHI": round(new_hhi, 4),
        "새비중": new_share
    }

In [ ]:
scenario_10 = diversified_rebalance(10)

if scenario_10 is None:
    print("10% 완화 조건을 만족하는 포트폴리오가 없습니다.")

else:
    diversified_result = portfolio[
        ["연령대", "현재비중"]
    ].copy()

    diversified_result["추천비중"] = (
        scenario_10["새비중"]
    ).round(1)

    diversified_result["변화(%p)"] = (
        diversified_result["추천비중"]
        - diversified_result["현재비중"]
    ).round(1)

    print(
        "2040 스트레스 :",
        scenario_10["전환전"],
        "→",
        scenario_10["전환후"]
    )

    print(
        "실제 위험완화율 :",
        scenario_10["실제완화율"],
        "%"
    )

    print(
        "총 고객비중 이동량 :",
        scenario_10["이동량"],
        "%p"
    )

    print(
        "연령집중도(HHI) :",
        scenario_10["기존HHI"],
        "→",
        scenario_10["추천HHI"]
    )

    diversified_result

In [ ]:
diversified_result

In [ ]:
diversified_result[
    [
        "연령대",
        "현재비중",
        "추천비중",
        "변화(%p)"
    ]
]

In [ ]:
print("증가한 연령대")

display(
    diversified_result[
        diversified_result["변화(%p)"] > 0
    ][
        [
            "연령대",
            "현재비중",
            "추천비중",
            "변화(%p)"
        ]
    ]
)

print("감소한 연령대")

display(
    diversified_result[
        diversified_result["변화(%p)"] < 0
    ][
        [
            "연령대",
            "현재비중",
            "추천비중",
            "변화(%p)"
        ]
    ]
)

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

def diversified_case(case_row, target_mitigation_pct=10):

    region = case_row["광역지역"]
    area = case_row["지역명"]
    code = case_row["업종코드"]
    industry = case_row["업종명"]

    p = age_pool[
        (age_pool["광역지역"] == region) &
        (age_pool["지역명"] == area) &
        (age_pool["업종코드"] == code)
    ][
        ["연령대", "이용금액비중", "2040 증감률"]
    ].copy()

    p = p.merge(
        benchmark[
            (benchmark["광역지역"] == region) &
            (benchmark["업종코드"] == code)
        ][
            ["연령대", "동일업종_Q25", "동일업종_Q75"]
        ],
        on="연령대",
        how="left"
    )

    if len(p) == 0:
        return None

    # 현재 비중 100% 정규화
    p["현재비중"] = (
        p["이용금액비중"]
        / p["이용금액비중"].sum()
        * 100
    )

    current = p["현재비중"].to_numpy()
    future = p["2040 증감률"].to_numpy()

    q25 = p["동일업종_Q25"].to_numpy()
    q75 = p["동일업종_Q75"].to_numpy()

    lower = np.minimum(current, q25)
    upper = np.maximum(current, q75)

    current_stress = np.sum(
        current * future / 100
    )

    current_hhi = np.sum(
        (current / 100) ** 2
    )

    target_stress = (
        current_stress
        + abs(current_stress)
        * target_mitigation_pct / 100
    )

    # 현재 고객구조에서 가능한 한 적게 변화
    def objective(x):
        return np.sum((x - current) ** 2)

    constraints = [
        {
            "type": "eq",
            "fun": lambda x: np.sum(x) - 100
        },
        {
            "type": "ineq",
            "fun": lambda x:
                np.sum(x * future / 100) - target_stress
        },
        {
            "type": "ineq",
            "fun": lambda x:
                current_hhi - np.sum((x / 100) ** 2)
        }
    ]

    result = minimize(
        objective,
        x0=current,
        method="SLSQP",
        bounds=list(zip(lower, upper)),
        constraints=constraints,
        options={
            "maxiter": 2000,
            "ftol": 1e-9
        }
    )

    if not result.success:
        return None

    new_share = result.x
    change = new_share - current

    after_stress = np.sum(
        new_share * future / 100
    )

    new_hhi = np.sum(
        (new_share / 100) ** 2
    )

    positive = change > 0.05

    increased_ages = (
        p.loc[positive, "연령대"]
        .astype(str)
        .tolist()
    )

    total_increase = change[positive].sum()

    senior_increase = change[
        p["연령대"].astype(str) == "60대이상"
    ].sum()

    senior_share = (
        senior_increase / total_increase * 100
        if total_increase > 0 else 0
    )

    return {
        "광역지역": region,
        "지역명": area,
        "업종명": industry,

        "증가연령수": int(positive.sum()),
        "증가연령": ", ".join(increased_ages),

        "60대증가분": round(max(senior_increase, 0), 2),
        "증가분중60대비율(%)": round(max(senior_share, 0), 1),

        "총이동량(%p)": round(
            np.abs(change).sum() / 2, 2
        ),

        "2040전환전": round(current_stress, 2),
        "2040전환후": round(after_stress, 2),

        "HHI전": round(current_hhi, 4),
        "HHI후": round(new_hhi, 4)
    }


diversified_results = []

for _, row in illusion_cases.iterrows():

    r = diversified_case(
        row,
        target_mitigation_pct=10
    )

    if r is not None:
        diversified_results.append(r)

diversified_all = pd.DataFrame(
    diversified_results
)

diversified_all

In [ ]:
print(
    "분석 성공 사례 :",
    len(diversified_all),
    "/",
    len(illusion_cases)
)

print(
    "평균 증가 연령대 수 :",
    round(
        diversified_all["증가연령수"].mean(),
        1
    )
)

print(
    "증가분 중 60대 평균 비율 :",
    round(
        diversified_all["증가분중60대비율(%)"].mean(),
        1
    ),
    "%"
)

print(
    "평균 고객비중 이동량 :",
    round(
        diversified_all["총이동량(%p)"].mean(),
        1
    ),
    "%p"
)

print(
    "HHI가 감소한 사례 :",
    (
        diversified_all["HHI후"]
        < diversified_all["HHI전"]
    ).sum(),
    "/",
    len(diversified_all)
)